# Phase 8 — Intelligent Testing Prioritization

## Objective
Develop a transparent, evidence-based framework to rank groundwater monitoring locations according to their need for further laboratory testing.

## Prioritization dimensions

1. Chemical risk:
   - Historical chemical limit exceedances.
   - Repeated exceedance evidence.
   - Severity of measured exceedances.

2. Temporal risk:
   - Statistically significant increasing chemical trends.
   - Robustness of trend evidence.
   - Historical trend consistency.

3. Monitoring gap:
   - Time since the latest recorded observation.
   - Historical sampling frequency.
   - Locations with long monitoring gaps.

4. Data uncertainty:
   - Missing chemical measurements.
   - Incomplete monitoring history.
   - Limited evidence for current groundwater conditions.

## Methodology
- Use provisional spatial groups as monitoring-location candidates.
- Preserve original records and laboratory measurements.
- Use the Phase 5 threshold-label register.
- Reuse verified Phase 7 location-specific trend results.
- Calculate transparent, explainable component scores.
- Produce ranked testing priorities and parameter-specific testing recommendations.
- Document evidence and limitations for every priority classification.

## Important limitations
- A high priority score indicates a greater need for investigation, not proof that groundwater is unsafe.
- Missing measurements are not treated as acceptable measurements.
- Provisional spatial groups are not confirmed well identities.
- Original administrative metadata errors must remain traceable.
- Applicable drinking-water threshold references must be verified before operational deployment.

## Expected outputs
- Location-level chemical risk register.
- Temporal risk register.
- Monitoring-gap and missing-data assessment.
- Composite testing-priority score.
- Ranked testing-priority recommendations.
- Explanation of the factors contributing to each priority.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate project root.
cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "reports").exists():
    PROJECT_ROOT = cwd
elif (
    (cwd.parent / "data").exists()
    and (cwd.parent / "reports").exists()
):
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate project root.")

MASTER_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "groundwater_master_processed.csv"
)

CLASSIFICATION_PATH = (
    PROJECT_ROOT
    / "reports"
    / "feature_engineering"
    / "classification_context_labels.csv"
)

LABEL_SUPPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "feature_engineering"
    / "classification_label_support.csv"
)

THRESHOLD_PATH = (
    PROJECT_ROOT
    / "reports"
    / "feature_engineering"
    / "threshold_reference_register.csv"
)

required_files = [
    MASTER_PATH,
    CLASSIFICATION_PATH,
    LABEL_SUPPORT_PATH,
    THRESHOLD_PATH
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

# Load original integrated records.
priority_master = pd.read_csv(
    MASTER_PATH,
    low_memory=False
)

# Load Phase 5 classification dataset.
priority_labels = pd.read_csv(
    CLASSIFICATION_PATH,
    low_memory=False
)

# Load supporting label and threshold registers.
label_support = pd.read_csv(LABEL_SUPPORT_PATH)
threshold_register = pd.read_csv(THRESHOLD_PATH)

print("PHASE 8 SOURCE DATASETS")
print("=" * 65)

print(f"Master records: {len(priority_master):,}")
print(f"Classification records: {len(priority_labels):,}")
print(f"Classification columns: {len(priority_labels.columns):,}")
print(f"Label-support rows: {len(label_support):,}")
print(f"Threshold-register rows: {len(threshold_register):,}")

assert priority_master["_record_id"].is_unique
assert priority_labels["_record_id"].is_unique

print("\nPHASE 5 CLASSIFICATION COLUMN NAMES")
for i, col in enumerate(priority_labels.columns):
    print(f"{i:2d}. {col}")

print("\nLABEL SUPPORT REGISTER")
display(label_support.head(15))

print("\nTHRESHOLD REFERENCE REGISTER")
display(threshold_register)

PHASE 8 SOURCE DATASETS
Master records: 174,635
Classification records: 174,635
Classification columns: 50
Label-support rows: 31
Threshold-register rows: 19

PHASE 5 CLASSIFICATION COLUMN NAMES
 0. _record_id
 1. _source_file
 2. _source_row
 3. _parsed_date
 4. _provisional_station_id
 5. _provisional_location_candidate_id
 6. _spatial_group_id
 7. _geo_grid_id
 8. _latitude
 9. _longitude
10. _coordinates_missing
11. _state_feature
12. _district_feature
13. _state_district_feature
14. _year_index
15. _month_sin
16. _month_cos
17. _dayofyear_sin
18. _dayofyear_cos
19. Potential of Hydrogen (pH)__exceeds_acceptable
20. Potential of Hydrogen (pH)__exceeds_permissible
21. Chloride (mg/L)__exceeds_acceptable
22. Chloride (mg/L)__exceeds_permissible
23. Magnesium (mg/L)__exceeds_acceptable
24. Magnesium (mg/L)__exceeds_permissible
25. Calcium (mg/L)__exceeds_acceptable
26. Calcium (mg/L)__exceeds_permissible
27. Total Hardness (mgCaCO3/L)__exceeds_acceptable
28. Total Hardness (mgCaCO3/L)

,Label,Available_Labels,Positive_Count,Negative_Count,Positive_Class_Percentage
0,Potential of Hydrogen (pH)__exceeds_acceptable,162719,10399,152320,6.391
1,Potential of Hydrogen (pH)__exceeds_permissible,162719,10399,152320,6.391
2,Chloride (mg/L)__exceeds_acceptable,162033,29757,132276,18.365
3,Chloride (mg/L)__exceeds_permissible,162033,5008,157025,3.091
4,Magnesium (mg/L)__exceeds_acceptable,150412,70219,80193,46.684
5,Magnesium (mg/L)__exceeds_permissible,150412,11423,138989,7.594
6,Calcium (mg/L)__exceeds_acceptable,149855,38451,111404,25.659
7,Calcium (mg/L)__exceeds_permissible,149855,4622,145233,3.084
8,Total Hardness (mgCaCO3/L)__exceeds_acceptable,144408,88597,55811,61.352
9,Total Hardness (mgCaCO3/L)__exceeds_permissible,144408,14789,129619,10.241



THRESHOLD REFERENCE REGISTER


,Parameter,Unit,Acceptable_Lower,Acceptable_Upper,Acceptable_Max,Permissible_Max,Rule,Status
0,Potential of Hydrogen (pH),pH,6.5,8.5,NaN,NaN,Range; no relaxation,Ready
1,Total Dissolved Solids (mg/L),mg/L,NaN,NaN,500.000,2000.0,Upper limit,Ready
2,Chloride (mg/L),mg/L,NaN,NaN,250.000,1000.0,Upper limit,Ready
3,Sulphate (mg/L),mg/L,NaN,NaN,200.000,400.0,Conditional on Mg <= 30 mg/L,Ready with condition
4,Total Hardness (mgCaCO3/L),mg/L as CaCO3,NaN,NaN,200.000,600.0,Upper limit,Ready
5,Calcium (mg/L),mg/L as Ca,NaN,NaN,75.000,200.0,Upper limit,Ready
6,Magnesium (mg/L),mg/L as Mg,NaN,NaN,30.000,100.0,Upper limit,Ready
7,Total Alkalinity (mg/L as CaCO3),mg/L as CaCO3,NaN,NaN,200.000,600.0,Upper limit,Ready
8,Arsenic (mg/L),mg/L as As,NaN,NaN,0.010,NaN,No relaxation,Provisional analyte match
9,Cadmium (mg/L),mg/L as Cd,NaN,NaN,0.003,NaN,No relaxation,Ready


In [2]:
known_metadata_columns = {
    "_record_id",
    "_source_file",
    "_source_row",
    "_parsed_date",
    "_provisional_station_id",
    "_provisional_location_candidate_id",
    "_spatial_group_id",
    "_geo_grid_id",
    "_latitude",
    "_longitude",
    "_coordinates_missing",
    "_state_feature",
    "_district_feature",
    "_state_district_feature",
    "_year_index",
    "_month_sin",
    "_month_cos",
    "_dayofyear_sin",
    "_dayofyear_cos"
}

classification_columns = [
    col for col in priority_labels.columns
    if col not in known_metadata_columns
]

label_schema_rows = []

for col in classification_columns:

    values = priority_labels[col]

    label_schema_rows.append({
        "Column": col,
        "Data type": str(values.dtype),
        "Non-missing": int(values.notna().sum()),
        "Missing": int(values.isna().sum()),
        "Unique values": int(values.nunique(dropna=True)),
        "Sample values": str(
            values.dropna().unique()[:6].tolist()
        )
    })

label_schema_df = pd.DataFrame(label_schema_rows)

print("CLASSIFICATION LABEL SCHEMA")
print("=" * 85)
display(label_schema_df)

print("\nPOSSIBLE PHASE 7 EXPORTS")
PHASE7_DIR = (
    PROJECT_ROOT
    / "reports"
    / "temporal_analysis"
    / "phase_7"
)

phase7_expected = [
    "phase7_location_trends_may_only.csv",
    "phase7_location_trends_all_months.csv",
    "phase7_location_review_register.csv",
    "phase7_significant_location_candidates.csv",
    "phase7_forecasting_decisions.csv"
]

for filename in phase7_expected:
    path = PHASE7_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'NOT FOUND'}")

CLASSIFICATION LABEL SCHEMA


,Column,Data type,Non-missing,Missing,Unique values,Sample values
0,Potential of Hydrogen (pH)__exceeds_acceptable,float64,162719,11916,2,"[0.0, 1.0]"
1,Potential of Hydrogen (pH)__exceeds_permissible,float64,162719,11916,2,"[0.0, 1.0]"
2,Chloride (mg/L)__exceeds_acceptable,float64,162033,12602,2,"[0.0, 1.0]"
3,Chloride (mg/L)__exceeds_permissible,float64,162033,12602,2,"[0.0, 1.0]"
4,Magnesium (mg/L)__exceeds_acceptable,float64,150412,24223,2,"[1.0, 0.0]"
5,Magnesium (mg/L)__exceeds_permissible,float64,150412,24223,2,"[0.0, 1.0]"
6,Calcium (mg/L)__exceeds_acceptable,float64,149855,24780,2,"[0.0, 1.0]"
7,Calcium (mg/L)__exceeds_permissible,float64,149855,24780,2,"[0.0, 1.0]"
8,Total Hardness (mgCaCO3/L)__exceeds_acceptable,float64,144408,30227,2,"[1.0, 0.0]"
9,Total Hardness (mgCaCO3/L)__exceeds_permissible,float64,144408,30227,2,"[0.0, 1.0]"



POSSIBLE PHASE 7 EXPORTS
phase7_location_trends_may_only.csv: FOUND
phase7_location_trends_all_months.csv: FOUND
phase7_location_review_register.csv: FOUND
phase7_significant_location_candidates.csv: FOUND
phase7_forecasting_decisions.csv: FOUND


In [3]:
PRIORITY_TARGETS = [
    "Potential of Hydrogen (pH)",
    "Electric Conductivity (μS/cm)",
    "Chloride (mg/L)",
    "Bicarbonate (mg/L)",
    "Sulphate (mg/L)",
    "Calcium (mg/L)",
    "Magnesium (mg/L)",
    "Sodium (mg/L)"
]

required_metadata = [
    "_record_id",
    "_spatial_group_id",
    "_state_feature",
    "_district_feature",
    "_latitude",
    "_longitude",
    "_coordinates_missing"
]

missing_metadata = [
    col for col in required_metadata
    if col not in priority_labels.columns
]

if missing_metadata:
    raise KeyError(f"Required Phase 5 metadata missing: {missing_metadata}")

# Merge spatial metadata onto the original records.
priority_metadata = priority_labels[
    required_metadata
].copy()

priority_work = priority_master.merge(
    priority_metadata,
    on="_record_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

unmatched = (priority_work["_merge"] != "both").sum()

assert unmatched == 0, f"{unmatched:,} records were not matched."

priority_work = priority_work.drop(columns=["_merge"]).copy()

# Parse acquisition date.
priority_work["_date"] = pd.to_datetime(
    priority_work["Data Acquisition Time"].astype(str).str.strip(),
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

priority_work["_year"] = priority_work["_date"].dt.year

# Establish dataset cutoff date.
DATASET_CUTOFF = priority_work["_date"].max()

print("DATASET CUTOFF DATE:", DATASET_CUTOFF)
print("Matched historical records:", f"{len(priority_work):,}")

# One row per provisional spatial group.
location_profile = (
    priority_work.dropna(subset=["_spatial_group_id"])
    .groupby("_spatial_group_id")
    .agg(
        State=("_state_feature", "first"),
        District=("_district_feature", "first"),
        Latitude=("_latitude", "median"),
        Longitude=("_longitude", "median"),
        First_observation=("_date", "min"),
        Latest_observation=("_date", "max"),
        Total_records=("_record_id", "size"),
        Distinct_years=("_year", "nunique"),
        Coordinate_missing_records=("_coordinates_missing", "sum")
    )
)

location_profile["Years_since_last_sample"] = (
    DATASET_CUTOFF - location_profile["Latest_observation"]
).dt.days / 365.25

# Count valid historical observations for each chemical parameter.
for target in PRIORITY_TARGETS:

    values = pd.to_numeric(
        priority_work[target], errors="coerce"
    )

    valid = (
        values.notna()
        & np.isfinite(values)
        & (values >= 0)
    )

    if target == "Potential of Hydrogen (pH)":
        valid &= (values > 0) & (values <= 14)

    target_counts = (
        priority_work.loc[valid]
        .groupby("_spatial_group_id")
        .size()
        .rename(f"Observed_count__{target}")
    )

    location_profile = location_profile.join(
        target_counts,
        how="left"
    )

    location_profile[f"Observed_count__{target}"] = (
        location_profile[f"Observed_count__{target}"]
        .fillna(0)
        .astype(int)
    )

print("\nLOCATION PROFILE")
print("=" * 70)
print(f"Provisional spatial groups: {len(location_profile):,}")
display(location_profile.head(10))

print("\nMONITORING RECENCY DISTRIBUTION")
display(
    location_profile["Years_since_last_sample"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
    .to_frame("Years since last sample")
    .round(2)
)

C:\Users\pokep\AppData\Local\Temp\ipykernel_28404\1797876769.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  priority_work = priority_master.merge(
C:\Users\pokep\AppData\Local\Temp\ipykernel_28404\1797876769.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  priority_work = priority_master.merge(


DATASET CUTOFF DATE: 2025-05-08 10:02:00
Matched historical records: 174,635

LOCATION PROFILE
Provisional spatial groups: 24,199


,State,District,Latitude,Longitude,First_observation,Latest_observation,Total_records,Distinct_years,Coordinate_missing_records,Years_since_last_sample,Observed_count__Potential of Hydrogen (pH),Observed_count__Electric Conductivity (μS/cm),Observed_count__Chloride (mg/L),Observed_count__Bicarbonate (mg/L),Observed_count__Sulphate (mg/L),Observed_count__Calcium (mg/L),Observed_count__Magnesium (mg/L),Observed_count__Sodium (mg/L)
_spatial_group_id,,,,,,,,,,,,,,,,,,
andaman and nicobar islands|north middle andaman|bakultala|site|12.50389|92.85917,andaman and nicobar islands,north middle andaman,12.503889,92.859167,2004-05-25 00:00:00,2021-05-30 08:00:00,8,8,0,3.939767,8,8,8,8,6,8,8,8
andaman and nicobar islands|north middle andaman|baratang (nilambur)|site|12.16694|92.76556,andaman and nicobar islands,north middle andaman,12.166944,92.765556,2004-05-25 00:00:00,2021-05-30 08:00:00,7,7,0,3.939767,7,7,7,7,5,7,7,7
andaman and nicobar islands|north middle andaman|basantipur|site|12.72917|92.88472,andaman and nicobar islands,north middle andaman,12.729167,92.884722,2004-05-25 00:00:00,2021-05-30 08:00:00,7,7,0,3.939767,7,7,7,7,5,7,7,7
andaman and nicobar islands|north middle andaman|betapur|site|12.62222|92.87278,andaman and nicobar islands,north middle andaman,12.622222,92.872778,2019-05-30 08:00:00,2021-05-30 08:00:00,3,3,0,3.939767,3,3,3,3,1,3,3,3
andaman and nicobar islands|north middle andaman|durgapur|site|13.23083|92.95528,andaman and nicobar islands,north middle andaman,13.230833,92.955278,2004-05-25 00:00:00,2018-05-25 00:00:00,5,5,0,6.954141,5,5,5,5,5,5,5,5
andaman and nicobar islands|north middle andaman|haribeck|site|12.98222|92.84556,andaman and nicobar islands,north middle andaman,12.982222,92.845556,2004-05-25 00:00:00,2007-05-25 00:00:00,3,3,0,17.954825,3,3,3,3,3,3,3,3
andaman and nicobar islands|north middle andaman|joypur|site|12.69917|92.88167,andaman and nicobar islands,north middle andaman,12.699167,92.881667,2018-05-25 00:00:00,2021-05-30 08:00:00,2,2,0,3.939767,2,2,2,2,2,2,2,2
andaman and nicobar islands|north middle andaman|kadamtala|site|12.34472|92.77889,andaman and nicobar islands,north middle andaman,12.344722,92.778889,2004-05-25 00:00:00,2021-05-30 08:00:00,8,8,0,3.939767,8,8,8,8,6,8,8,8
andaman and nicobar islands|north middle andaman|kausalyanagar|site|12.39528|92.78222,andaman and nicobar islands,north middle andaman,12.395278,92.782222,2004-05-25 00:00:00,2016-05-25 00:00:00,4,4,0,8.952772,4,4,4,4,4,4,4,4



MONITORING RECENCY DISTRIBUTION


,Years since last sample
count,24199.00
mean,8.11
std,6.23
min,0.00
25%,3.89
50%,6.95
75%,10.95
90%,18.95
95%,21.95
max,24.95


In [4]:
# Freeze the reference date for reproducibility.
AS_OF_DATE = pd.Timestamp("2026-11-23")

# Recalculate monitoring gaps relative to the actual assessment date.
location_profile["Years_since_last_sample"] = (
    AS_OF_DATE - location_profile["Latest_observation"]
).dt.days / 365.25

print("MONITORING RECENCY — AS OF", AS_OF_DATE.date())
print("=" * 70)

display(
    location_profile["Years_since_last_sample"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .to_frame("Years since last sample")
    .round(2)
)

print("\nLATEST DATE IN THE DATASET:")
print(priority_work["_date"].max())

print("\nALL 31 CLASSIFICATION LABELS")
print("=" * 70)

label_columns = [
    col for col in priority_labels.columns
    if col not in known_metadata_columns
]

for i, col in enumerate(label_columns, start=1):
    print(f"{i:2d}. {col}")

print("\nCOMPLETE LABEL SUPPORT REGISTER")
print("=" * 70)
print(label_support.to_string(index=False))

print("\nCOMPLETE THRESHOLD REFERENCE REGISTER")
print("=" * 70)
print(threshold_register.to_string(index=False))

MONITORING RECENCY — AS OF 2026-11-23


,Years since last sample
count,24199.00
mean,9.65
std,6.23
min,1.54
10%,3.56
25%,5.44
50%,8.50
75%,12.50
90%,20.50
95%,23.50



LATEST DATE IN THE DATASET:
2025-05-08 10:02:00

ALL 31 CLASSIFICATION LABELS
 1. Potential of Hydrogen (pH)__exceeds_acceptable
 2. Potential of Hydrogen (pH)__exceeds_permissible
 3. Chloride (mg/L)__exceeds_acceptable
 4. Chloride (mg/L)__exceeds_permissible
 5. Magnesium (mg/L)__exceeds_acceptable
 6. Magnesium (mg/L)__exceeds_permissible
 7. Calcium (mg/L)__exceeds_acceptable
 8. Calcium (mg/L)__exceeds_permissible
 9. Total Hardness (mgCaCO3/L)__exceeds_acceptable
10. Total Hardness (mgCaCO3/L)__exceeds_permissible
11. Sulphate (mg/L)__exceeds_acceptable
12. Sulphate (mg/L)__exceeds_permissible
13. Total Alkalinity (mg/L as CaCO3)__exceeds_acceptable
14. Total Alkalinity (mg/L as CaCO3)__exceeds_permissible
15. Total Dissolved Solids (mg/L)__exceeds_acceptable
16. Total Dissolved Solids (mg/L)__exceeds_permissible
17. Arsenic (mg/L)__exceeds_acceptable
18. Arsenic (mg/L)__exceeds_permissible
19. Copper (mg/L)__exceeds_acceptable
20. Copper (mg/L)__exceeds_permissible
21. Lead (m

In [5]:
# Match each classification label to its threshold-reference parameter.
label_mapping_rows = []

support_lookup = label_support.set_index("Label")

threshold_lookup = threshold_register.set_index("Parameter")

for col in label_columns:

    if "__" not in col:
        continue

    parameter, rule = col.split("__", 1)

    if parameter in threshold_lookup.index:
        threshold_status = threshold_lookup.loc[
            parameter, "Status"
        ]
        threshold_rule = threshold_lookup.loc[
            parameter, "Rule"
        ]
    elif parameter == "Iron_Manganese":
        threshold_status = "Deferred for joint-rule review"
        threshold_rule = "Combined iron-manganese condition"
    else:
        threshold_status = "No matching threshold register entry"
        threshold_rule = "Review required"

    support_row = (
        support_lookup.loc[col]
        if col in support_lookup.index
        else None
    )

    label_mapping_rows.append({
        "Label": col,
        "Parameter": parameter,
        "Rule": rule,
        "Threshold_status": threshold_status,
        "Threshold_rule": threshold_rule,
        "Available_labels": (
            support_row["Available_Labels"]
            if support_row is not None else np.nan
        ),
        "Positive_count": (
            support_row["Positive_Count"]
            if support_row is not None else np.nan
        )
    })

label_mapping_df = pd.DataFrame(label_mapping_rows)

print("THRESHOLD-TO-LABEL MAPPING")
print("=" * 90)
display(label_mapping_df)

print("\nTHRESHOLD STATUS COUNTS")
display(
    label_mapping_df.groupby("Threshold_status")
    .agg(
        Label_columns=("Label", "count"),
        Positive_observations=("Positive_count", "sum")
    )
)

THRESHOLD-TO-LABEL MAPPING


,Label,Parameter,Rule,Threshold_status,Threshold_rule,Available_labels,Positive_count
0,Potential of Hydrogen (pH)__exceeds_acceptable,Potential of Hydrogen (pH),exceeds_acceptable,Ready,Range; no relaxation,162719.0,10399.0
1,Potential of Hydrogen (pH)__exceeds_permissible,Potential of Hydrogen (pH),exceeds_permissible,Ready,Range; no relaxation,162719.0,10399.0
2,Chloride (mg/L)__exceeds_acceptable,Chloride (mg/L),exceeds_acceptable,Ready,Upper limit,162033.0,29757.0
3,Chloride (mg/L)__exceeds_permissible,Chloride (mg/L),exceeds_permissible,Ready,Upper limit,162033.0,5008.0
4,Magnesium (mg/L)__exceeds_acceptable,Magnesium (mg/L),exceeds_acceptable,Ready,Upper limit,150412.0,70219.0
5,Magnesium (mg/L)__exceeds_permissible,Magnesium (mg/L),exceeds_permissible,Ready,Upper limit,150412.0,11423.0
6,Calcium (mg/L)__exceeds_acceptable,Calcium (mg/L),exceeds_acceptable,Ready,Upper limit,149855.0,38451.0
7,Calcium (mg/L)__exceeds_permissible,Calcium (mg/L),exceeds_permissible,Ready,Upper limit,149855.0,4622.0
8,Total Hardness (mgCaCO3/L)__exceeds_acceptable,Total Hardness (mgCaCO3/L),exceeds_acceptable,Ready,Upper limit,144408.0,88597.0
9,Total Hardness (mgCaCO3/L)__exceeds_permissible,Total Hardness (mgCaCO3/L),exceeds_permissible,Ready,Upper limit,144408.0,14789.0



THRESHOLD STATUS COUNTS


,Label_columns,Positive_observations
Threshold_status,,
Deferred for joint-rule review,1,106.0
Provisional analyte match,2,114.0
Ready,26,343536.0
Ready with condition,2,26578.0


In [6]:
# Eight primary/secondary chemical-risk parameters.
PRIORITY_CHEMICAL_SPECS = [
    {
        "key": "pH",
        "parameter": "Potential of Hydrogen (pH)",
        "mode": "no_relaxation"
    },
    {
        "key": "Chloride",
        "parameter": "Chloride (mg/L)",
        "mode": "standard"
    },
    {
        "key": "Magnesium",
        "parameter": "Magnesium (mg/L)",
        "mode": "standard"
    },
    {
        "key": "Calcium",
        "parameter": "Calcium (mg/L)",
        "mode": "standard"
    },
    {
        "key": "Hardness",
        "parameter": "Total Hardness (mgCaCO3/L)",
        "mode": "standard"
    },
    {
        "key": "Sulphate",
        "parameter": "Sulphate (mg/L)",
        "mode": "conditional"
    },
    {
        "key": "Alkalinity",
        "parameter": "Total Alkalinity (mg/L as CaCO3)",
        "mode": "standard"
    },
    {
        "key": "TDS",
        "parameter": "Total Dissolved Solids (mg/L)",
        "mode": "standard"
    }
]

# Trace and other sparse indicators remain separate from the primary score.
TRACE_ALERT_SPECS = {
    "Arsenic": {
        "columns": [
            "Arsenic (mg/L)__exceeds_acceptable",
            "Arsenic (mg/L)__exceeds_permissible"
        ],
        "status": "Provisional analyte match"
    },
    "Copper": {
        "columns": [
            "Copper (mg/L)__exceeds_acceptable",
            "Copper (mg/L)__exceeds_permissible"
        ],
        "status": "Ready, sparse observations"
    },
    "Lead": {
        "columns": [
            "Lead (mg/L)__exceeds_acceptable",
            "Lead (mg/L)__exceeds_permissible"
        ],
        "status": "Ready, sparse observations"
    },
    "Zinc": {
        "columns": [
            "Zinc (mg/L)__exceeds_acceptable",
            "Zinc (mg/L)__exceeds_permissible"
        ],
        "status": "Ready, sparse observations"
    },
    "Nickel": {
        "columns": [
            "Nickel (mg/L)__exceeds_acceptable",
            "Nickel (mg/L)__exceeds_permissible"
        ],
        "status": "Ready, sparse observations"
    },
    "Cadmium": {
        "columns": [
            "Cadmium (mg/L)__exceeds_acceptable",
            "Cadmium (mg/L)__exceeds_permissible"
        ],
        "status": "Ready, sparse observations"
    },
    "Chromium": {
        "columns": [
            "Chromium (mg/L)__exceeds_acceptable",
            "Chromium (mg/L)__exceeds_permissible"
        ],
        "status": "Ready, sparse observations"
    },
    "Iron_Manganese": {
        "columns": [
            "Iron_Manganese__combined_exceeds_0_3"
        ],
        "status": "Deferred for joint-rule review"
    }
}

# Collect exactly the classification columns needed.
required_label_columns = []

for spec in PRIORITY_CHEMICAL_SPECS:
    required_label_columns.extend([
        f"{spec['parameter']}__exceeds_acceptable",
        f"{spec['parameter']}__exceeds_permissible"
    ])

for spec in TRACE_ALERT_SPECS.values():
    required_label_columns.extend(spec["columns"])

required_label_columns = list(dict.fromkeys(required_label_columns))

missing_label_columns = [
    col for col in required_label_columns
    if col not in priority_labels.columns
]

if missing_label_columns:
    raise KeyError(f"Classification labels missing: {missing_label_columns}")

# Merge existing classification labels onto the historical records.
priority_evidence = priority_work.merge(
    priority_labels[["_record_id"] + required_label_columns],
    on="_record_id",
    how="left",
    validate="one_to_one"
)

assert len(priority_evidence) == len(priority_master)
assert priority_evidence["_record_id"].is_unique

# Calculate observation-level chemical severity.
global_severity_rates = {}
severity_columns = []
severity_support_rows = []

for spec in PRIORITY_CHEMICAL_SPECS:

    key = spec["key"]
    parameter = spec["parameter"]

    acceptable_col = f"{parameter}__exceeds_acceptable"
    permissible_col = f"{parameter}__exceeds_permissible"

    acceptable = pd.to_numeric(
        priority_evidence[acceptable_col],
        errors="coerce"
    )

    permissible = pd.to_numeric(
        priority_evidence[permissible_col],
        errors="coerce"
    )

    valid = acceptable.notna()

    severity = pd.Series(
        np.nan,
        index=priority_evidence.index,
        dtype=float
    )

    if spec["mode"] == "no_relaxation":

        # No separate relaxed permissible category.
        severity.loc[valid] = (
            acceptable.loc[valid] == 1
        ).astype(float)

    else:

        severity.loc[valid] = 0.0

        severe = valid & permissible.eq(1)

        moderate = (
            valid
            & acceptable.eq(1)
            & ~permissible.eq(1)
        )

        severity.loc[moderate] = 0.5
        severity.loc[severe] = 1.0

    severity_col = f"_severity__{key}"
    priority_evidence[severity_col] = severity

    severity_columns.append(severity_col)

    # Empirical national prior for this parameter.
    global_rate = severity.mean()

    if pd.isna(global_rate):
        raise ValueError(f"No valid severity observations for {parameter}")

    global_severity_rates[key] = float(global_rate)

    severity_support_rows.append({
        "Parameter": parameter,
        "Valid observations": int(valid.sum()),
        "Moderate observations": int(
            (severity == 0.5).sum()
        ),
        "Severe observations": int(
            (severity == 1.0).sum()
        ),
        "National mean severity": global_rate,
        "Scoring mode": spec["mode"]
    })

    # Identify incomplete conditional sulphate assessments.
    if spec["mode"] == "conditional":
        priority_evidence[
            f"_conditional_unknown__{key}"
        ] = (
            valid
            & acceptable.eq(1)
            & permissible.isna()
        )

# Create separate flags for sparse trace indicators.
trace_row_flag_columns = []

for key, spec in TRACE_ALERT_SPECS.items():

    cols = spec["columns"]

    values = priority_evidence[cols].apply(
        pd.to_numeric,
        errors="coerce"
    )

    measured_col = f"_trace_measured__{key}"
    alert_col = f"_trace_alert__{key}"

    priority_evidence[measured_col] = values.notna().any(axis=1)
    priority_evidence[alert_col] = values.eq(1).any(axis=1)

    trace_row_flag_columns.append(alert_col)

print("CHEMICAL SEVERITY SUPPORT")
print("=" * 80)
display(pd.DataFrame(severity_support_rows).round(4))

print("\nSeparate trace and deferred evidence flags created:")
print(list(TRACE_ALERT_SPECS))

CHEMICAL SEVERITY SUPPORT


,Parameter,Valid observations,Moderate observations,Severe observations,National mean severity,Scoring mode
0,Potential of Hydrogen (pH),162719,0,10399,0.0639,no_relaxation
1,Chloride (mg/L),162033,24749,5008,0.1073,standard
2,Magnesium (mg/L),150412,58796,11423,0.2714,standard
3,Calcium (mg/L),149855,33829,4622,0.1437,standard
4,Total Hardness (mgCaCO3/L),144408,73808,14789,0.3580,standard
5,Sulphate (mg/L),126326,1026,12776,0.1052,conditional
6,Total Alkalinity (mg/L as CaCO3),79936,39345,2751,0.2805,standard
7,Total Dissolved Solids (mg/L),30151,11009,1998,0.2488,standard



Separate trace and deferred evidence flags created:
['Arsenic', 'Copper', 'Lead', 'Zinc', 'Nickel', 'Cadmium', 'Chromium', 'Iron_Manganese']


In [7]:
PRIOR_STRENGTH = 5

# Use the existing Phase 3/Phase 5 location profile.
priority_profile = location_profile.copy()

chemical_score_columns = []
chemical_count_columns = []

for spec in PRIORITY_CHEMICAL_SPECS:

    key = spec["key"]
    severity_col = f"_severity__{key}"

    grouped = (
        priority_evidence
        .groupby("_spatial_group_id")[severity_col]
        .agg(
            Severity_sum="sum",
            Valid_count="count"
        )
        .reindex(priority_profile.index)
    )

    severity_sum = grouped["Severity_sum"].fillna(0)
    valid_count = grouped["Valid_count"].fillna(0)

    global_rate = global_severity_rates[key]

    posterior_rate = (
        severity_sum + PRIOR_STRENGTH * global_rate
    ) / (
        valid_count + PRIOR_STRENGTH
    )

    # No valid evidence at a location remains missing.
    posterior_rate.loc[valid_count == 0] = np.nan

    score_col = f"Smoothed_severity__{key}"
    count_col = f"Valid_label_observations__{key}"

    priority_profile[score_col] = posterior_rate
    priority_profile[count_col] = valid_count.astype(int)

    chemical_score_columns.append(score_col)
    chemical_count_columns.append(count_col)

# Average across chemical parameters with actual evidence.
priority_profile["Chemical_parameters_tested"] = (
    priority_profile[chemical_count_columns]
    .gt(0)
    .sum(axis=1)
)

priority_profile["Chemical_valid_observations"] = (
    priority_profile[chemical_count_columns]
    .sum(axis=1)
)

national_prior = np.mean(
    list(global_severity_rates.values())
)

priority_profile["Chemical_risk_score"] = (
    priority_profile[chemical_score_columns]
    .mean(axis=1)
    .mul(100)
    .fillna(national_prior * 100)
)

# Chemical measurement coverage across all eight scored parameters.
priority_profile["Chemical_coverage_fraction"] = (
    priority_profile["Chemical_valid_observations"]
    / (
        priority_profile["Total_records"]
        * len(PRIORITY_CHEMICAL_SPECS)
    )
).clip(0, 1)

missingness_score = (
    1 - priority_profile["Chemical_coverage_fraction"]
) * 100

history_completeness = (
    priority_profile["Distinct_years"] / 5
).clip(0, 1)

history_uncertainty_score = (
    1 - history_completeness
) * 100

priority_profile["Data_uncertainty_score"] = (
    0.70 * missingness_score
    + 0.30 * history_uncertainty_score
)

# Aggregate sparse trace alerts separately.
trace_support_rows = []

for key, spec in TRACE_ALERT_SPECS.items():

    measured_col = f"_trace_measured__{key}"
    alert_col = f"_trace_alert__{key}"

    grouped = (
        priority_evidence
        .groupby("_spatial_group_id")
        .agg(
            measured_count=(measured_col, "sum"),
            alert_count=(alert_col, "sum")
        )
        .reindex(priority_profile.index)
    )

    priority_profile[f"Trace_measured__{key}"] = (
        grouped["measured_count"].fillna(0).astype(int)
    )

    priority_profile[f"Trace_alerts__{key}"] = (
        grouped["alert_count"].fillna(0).astype(int)
    )

    trace_support_rows.append({
        "Parameter": key,
        "Threshold status": spec["status"],
        "Locations measured": int(
            (grouped["measured_count"].fillna(0) > 0).sum()
        ),
        "Locations with historical alerts": int(
            (grouped["alert_count"].fillna(0) > 0).sum()
        )
    })

priority_profile["Any_sparse_trace_alert"] = (
    priority_profile[
        [f"Trace_alerts__{key}" for key in TRACE_ALERT_SPECS]
    ].sum(axis=1) > 0
)

print("LOCATION-LEVEL CHEMICAL RISK")
print("=" * 70)

display(
    priority_profile[
        [
            "Chemical_risk_score",
            "Chemical_parameters_tested",
            "Chemical_valid_observations",
            "Chemical_coverage_fraction",
            "Data_uncertainty_score",
            "Years_since_last_sample",
            "Any_sparse_trace_alert"
        ]
    ].describe().round(3)
)

print("\nSPARSE TRACE ALERT SUMMARY")
display(pd.DataFrame(trace_support_rows))

LOCATION-LEVEL CHEMICAL RISK


,Chemical_risk_score,Chemical_parameters_tested,Chemical_valid_observations,Chemical_coverage_fraction,Data_uncertainty_score,Years_since_last_sample
count,24199.000,24199.000,24199.000,24199.000,24199.000,24199.000
mean,18.772,7.101,41.565,0.709,27.719,9.651
std,6.927,1.423,35.497,0.180,16.321,6.227
min,4.565,0.000,0.000,0.000,3.500,1.541
25%,14.576,7.000,12.000,0.625,15.641,5.437
50%,17.731,7.000,31.000,0.750,24.000,8.498
75%,21.520,8.000,60.000,0.833,35.000,12.498
max,60.766,8.000,159.000,1.000,94.000,26.497



SPARSE TRACE ALERT SUMMARY


,Parameter,Threshold status,Locations measured,Locations with historical alerts
0,Arsenic,Provisional analyte match,7929,56
1,Copper,"Ready, sparse observations",649,2
2,Lead,"Ready, sparse observations",648,8
3,Zinc,"Ready, sparse observations",648,0
4,Nickel,"Ready, sparse observations",649,1
5,Cadmium,"Ready, sparse observations",649,0
6,Chromium,"Ready, sparse observations",649,0
7,Iron_Manganese,Deferred for joint-rule review,647,92


In [8]:
# Load May-only Phase 7 location trends.
MAY_TREND_PATH = (
    PHASE7_DIR / "phase7_location_trends_may_only.csv"
)

if not MAY_TREND_PATH.exists():
    raise FileNotFoundError(MAY_TREND_PATH)

may_priority_trends = pd.read_csv(
    MAY_TREND_PATH,
    low_memory=False
)

required_trend_cols = [
    "Target",
    "_spatial_group_id",
    "Theil_Sen_slope_per_year",
    "BH_q_value"
]

missing_trend_cols = [
    col for col in required_trend_cols
    if col not in may_priority_trends.columns
]

if missing_trend_cols:
    raise KeyError(f"Phase 7 trend columns missing: {missing_trend_cols}")

may_priority_trends["_spatial_group_id"] = (
    may_priority_trends["_spatial_group_id"].astype(str)
)

# Keep statistically significant increasing trends for the eight core
# temporal parameters.
increasing_trends = may_priority_trends[
    may_priority_trends["Target"].isin(PRIORITY_TARGETS)
    & (may_priority_trends["BH_q_value"] < 0.05)
    & (may_priority_trends["Theil_Sen_slope_per_year"] > 0)
].copy()

trend_location_summary = (
    increasing_trends.groupby("_spatial_group_id")
    .agg(
        Significant_increasing_trend_count=("Target", "nunique"),
        Increasing_trend_parameters=(
            "Target",
            lambda values: ", ".join(sorted(set(values)))
        )
    )
)

priority_profile = priority_profile.join(
    trend_location_summary
)

priority_profile["Significant_increasing_trend_count"] = (
    priority_profile["Significant_increasing_trend_count"]
    .fillna(0)
    .astype(int)
)

priority_profile["Increasing_trend_parameters"] = (
    priority_profile["Increasing_trend_parameters"]
    .fillna("")
)

# Component 1: Chemical evidence score, already 0–100.
priority_profile["Chemical_component_score"] = (
    priority_profile["Chemical_risk_score"]
)

# Component 2: Number of significant increasing parameters,
# expressed as a fraction of the eight monitored parameters.
priority_profile["Temporal_component_score"] = (
    priority_profile["Significant_increasing_trend_count"]
    / len(PRIORITY_TARGETS)
    * 100
)

# Component 3: Monitoring age, capped at ten years.
priority_profile["Monitoring_gap_score"] = (
    priority_profile["Years_since_last_sample"]
    .div(10)
    .clip(0, 1)
    * 100
)

# Component 4: Missing data and limited monitoring history.
priority_profile["Uncertainty_component_score"] = (
    priority_profile["Data_uncertainty_score"]
)

# Transparent pilot weights.
PRIORITY_WEIGHTS = {
    "Chemical_component_score": 0.40,
    "Temporal_component_score": 0.25,
    "Monitoring_gap_score": 0.20,
    "Uncertainty_component_score": 0.15
}

assert abs(sum(PRIORITY_WEIGHTS.values()) - 1.0) < 1e-9

priority_profile["Testing_priority_score"] = sum(
    priority_profile[column] * weight
    for column, weight in PRIORITY_WEIGHTS.items()
)

# Explanation fields.
def explain_priority(row):

    reasons = []

    if row["Chemical_component_score"] >= 50:
        reasons.append("Elevated historical chemical exceedance evidence")

    if row["Significant_increasing_trend_count"] > 0:
        reasons.append(
            "Increasing trend: "
            + row["Increasing_trend_parameters"]
        )

    if row["Years_since_last_sample"] >= 10:
        reasons.append("No sample recorded in at least ten years")

    if row["Data_uncertainty_score"] >= 70:
        reasons.append("High chemical-data uncertainty")

    if row["Any_sparse_trace_alert"]:
        reasons.append("Historical sparse trace-parameter alert")

    if not reasons:
        reasons.append("Routine monitoring evidence; maintain surveillance")

    return "; ".join(reasons)

priority_profile["Priority_explanation"] = (
    priority_profile.apply(explain_priority, axis=1)
)

# Rank all provisional locations.
priority_profile["Priority_percentile"] = (
    priority_profile["Testing_priority_score"]
    .rank(method="average", pct=True)
    * 100
)

priority_profile["Priority_rank"] = (
    priority_profile["Testing_priority_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

priority_profile["Priority_tier"] = np.select(
    [
        priority_profile["Priority_percentile"] >= 90,
        priority_profile["Priority_percentile"] >= 75
    ],
    [
        "P1 — Highest 10%",
        "P2 — Next 15%"
    ],
    default="P3 — Remaining locations"
)

print("INTELLIGENT TESTING PRIORITIZATION — INITIAL RESULTS")
print("=" * 85)

print("Locations ranked:", f"{len(priority_profile):,}")
print("Weights:", PRIORITY_WEIGHTS)

display(
    priority_profile.sort_values(
        "Testing_priority_score",
        ascending=False
    )[
        [
            "State",
            "District",
            "Latitude",
            "Longitude",
            "Testing_priority_score",
            "Priority_rank",
            "Priority_tier",
            "Chemical_component_score",
            "Temporal_component_score",
            "Monitoring_gap_score",
            "Uncertainty_component_score",
            "Chemical_parameters_tested",
            "Significant_increasing_trend_count",
            "Priority_explanation"
        ]
    ].head(20).round(3)
)

print("\nPRIORITY TIER DISTRIBUTION")
display(
    priority_profile["Priority_tier"]
    .value_counts()
    .to_frame("Locations")
)

INTELLIGENT TESTING PRIORITIZATION — INITIAL RESULTS
Locations ranked: 24,199
Weights: {'Chemical_component_score': 0.4, 'Temporal_component_score': 0.25, 'Monitoring_gap_score': 0.2, 'Uncertainty_component_score': 0.15}


,State,District,Latitude,Longitude,Testing_priority_score,Priority_rank,Priority_tier,Chemical_component_score,Temporal_component_score,Monitoring_gap_score,Uncertainty_component_score,Chemical_parameters_tested,Significant_increasing_trend_count,Priority_explanation
_spatial_group_id,,,,,,,,,,,,,,
gujarat|kachchh|lilpur|site|23.51111|70.65,gujarat,kachchh,23.511,70.650,44.267,1,P1 — Highest 10%,55.032,12.5,84.983,14.219,8,1,Elevated historical chemical exceedance eviden...
odisha|malkangiri|m.v.79|site|17.985|81.595,odisha,malkangiri,17.985,81.595,44.229,2,P1 — Highest 10%,31.885,0.0,100.000,76.500,2,0,No sample recorded in at least ten years; High...
maharashtra|jalgaon|sadavan bk.|site|20.85278|75.1,maharashtra,jalgaon,20.853,75.100,43.369,3,P1 — Highest 10%,50.767,0.0,100.000,20.417,7,0,Elevated historical chemical exceedance eviden...
gujarat|kachchh|gagodar|site|23.41667|70.8,gujarat,kachchh,23.417,70.800,43.274,4,P1 — Highest 10%,52.717,0.0,100.000,14.583,7,0,Elevated historical chemical exceedance eviden...
delhi|south west|dwarka sect 12 deep|site|28.59722|77.04639,delhi,south west,28.597,77.046,43.030,5,P1 — Highest 10%,25.606,0.0,100.000,85.250,1,0,No sample recorded in at least ten years; High...
karnataka|davanagere|jagalur|site|14.58333|76.41667,karnataka,davanagere,14.583,76.417,43.030,6,P1 — Highest 10%,25.606,0.0,100.000,85.250,1,0,No sample recorded in at least ten years; High...
gujarat|junagadh|maktupur1|site|21.14306|70.09583,gujarat,junagadh,21.143,70.096,42.804,7,P1 — Highest 10%,50.448,0.0,100.000,17.500,8,0,Elevated historical chemical exceedance eviden...
karnataka|gadag|mallapur|site|15.675|75.63333,karnataka,gadag,15.675,75.633,42.786,8,P1 — Highest 10%,25.606,0.0,100.000,83.625,1,0,No sample recorded in at least ten years; High...
andhra pradesh|prakasam|parchuru|site|15.96667|80.275,andhra pradesh,prakasam,15.967,80.275,42.716,9,P1 — Highest 10%,46.010,0.0,100.000,28.750,7,0,No sample recorded in at least ten years



PRIORITY TIER DISTRIBUTION


,Locations
Priority_tier,
P3 — Remaining locations,18149
P2 — Next 15%,3629
P1 — Highest 10%,2421


In [9]:
# Correct the conditional sulphate rule.
SULPHATE_ACCEPTABLE = "Sulphate (mg/L)__exceeds_acceptable"
SULPHATE_PERMISSIBLE = "Sulphate (mg/L)__exceeds_permissible"

sulphate_acceptable = pd.to_numeric(
    priority_evidence[SULPHATE_ACCEPTABLE],
    errors="coerce"
)

sulphate_permissible = pd.to_numeric(
    priority_evidence[SULPHATE_PERMISSIBLE],
    errors="coerce"
)

# An acceptable-limit exceedance with unavailable permissible
# assessment is conditionally unresolved.
sulphate_unknown = (
    sulphate_acceptable.eq(1)
    & sulphate_permissible.isna()
)

sulphate_severity_col = "_severity__Sulphate"

priority_evidence.loc[
    sulphate_unknown,
    sulphate_severity_col
] = np.nan

priority_evidence["_conditional_unknown__Sulphate"] = (
    sulphate_unknown
)

print("SULPHATE CONDITIONAL-RULE CORRECTION")
print("-" * 60)
print("Conditionally unresolved observations:", int(sulphate_unknown.sum()))
print(
    "Sulphate observations included in severity:",
    int(priority_evidence[sulphate_severity_col].count())
)

# Recalculate the location-level chemical severity with the corrected labels.
for spec in PRIORITY_CHEMICAL_SPECS:

    key = spec["key"]
    severity_col = f"_severity__{key}"

    grouped = (
        priority_evidence
        .groupby("_spatial_group_id")[severity_col]
        .agg(["sum", "count"])
        .reindex(priority_profile.index)
    )

    severity_sum = grouped["sum"].fillna(0)
    valid_count = grouped["count"].fillna(0)

    global_rate = (
        priority_evidence[severity_col].mean()
    )

    global_severity_rates[key] = float(global_rate)

    posterior_rate = (
        severity_sum + PRIOR_STRENGTH * global_rate
    ) / (
        valid_count + PRIOR_STRENGTH
    )

    posterior_rate.loc[valid_count == 0] = np.nan

    priority_profile[f"Smoothed_severity__{key}"] = (
        posterior_rate
    )

    priority_profile[
        f"Valid_label_observations__{key}"
    ] = valid_count.astype(int)

# Count unresolved sulphate cases by location.
unknown_by_location = (
    priority_evidence
    .groupby("_spatial_group_id")[
        "_conditional_unknown__Sulphate"
    ]
    .sum()
    .reindex(priority_profile.index)
    .fillna(0)
    .astype(int)
)

priority_profile["Sulphate_conditional_unresolved_records"] = (
    unknown_by_location
)

# Recalculate chemical coverage.
priority_profile["Chemical_parameters_tested"] = (
    priority_profile[chemical_count_columns]
    .gt(0)
    .sum(axis=1)
)

priority_profile["Chemical_valid_observations"] = (
    priority_profile[chemical_count_columns].sum(axis=1)
)

priority_profile["Chemical_coverage_fraction"] = (
    priority_profile["Chemical_valid_observations"]
    / (
        priority_profile["Total_records"]
        * len(PRIORITY_CHEMICAL_SPECS)
    )
).clip(0, 1)

# Recalculate uncertainty.
missingness_score = (
    1 - priority_profile["Chemical_coverage_fraction"]
) * 100

history_completeness = (
    priority_profile["Distinct_years"] / 5
).clip(0, 1)

priority_profile["Data_uncertainty_score"] = (
    0.70 * missingness_score
    + 0.30 * (1 - history_completeness) * 100
)

# Recalculate national prior and chemical risk component.
national_prior = np.mean(
    list(global_severity_rates.values())
)

priority_profile["Chemical_risk_score"] = (
    priority_profile[chemical_score_columns]
    .mean(axis=1)
    .mul(100)
    .fillna(national_prior * 100)
)

priority_profile["Chemical_component_score"] = (
    priority_profile["Chemical_risk_score"]
)

# Preserve existing Phase 7 temporal evidence and recalculate total.
priority_profile["Temporal_component_score"] = (
    priority_profile["Significant_increasing_trend_count"]
    / len(PRIORITY_TARGETS)
    * 100
)

priority_profile["Monitoring_gap_score"] = (
    priority_profile["Years_since_last_sample"]
    .div(10)
    .clip(0, 1)
    * 100
)

priority_profile["Uncertainty_component_score"] = (
    priority_profile["Data_uncertainty_score"]
)

priority_profile["Testing_priority_score"] = sum(
    priority_profile[column] * weight
    for column, weight in PRIORITY_WEIGHTS.items()
)

# Make the explanation explicit about missing evidence.
def explain_priority_corrected(row):

    reasons = []

    if row["Chemical_parameters_tested"] == 0:
        reasons.append(
            "No valid measurements for the eight primary scored parameters"
        )
    elif row["Chemical_component_score"] >= 50:
        reasons.append(
            "Elevated historical chemical exceedance evidence"
        )

    if row["Significant_increasing_trend_count"] > 0:
        reasons.append(
            "Increasing trend: "
            + row["Increasing_trend_parameters"]
        )

    if row["Years_since_last_sample"] >= 10:
        reasons.append(
            "No sample recorded in at least ten years"
        )

    if row["Data_uncertainty_score"] >= 70:
        reasons.append("High chemical-data uncertainty")

    if row["Sulphate_conditional_unresolved_records"] > 0:
        reasons.append(
            "Sulphate conditional assessment unresolved"
        )

    if row["Any_sparse_trace_alert"]:
        reasons.append(
            "Historical sparse trace-parameter alert; review separately"
        )

    if not reasons:
        reasons.append(
            "Routine monitoring evidence; maintain surveillance"
        )

    return "; ".join(reasons)


priority_profile["Priority_explanation"] = (
    priority_profile.apply(
        explain_priority_corrected,
        axis=1
    )
)

# Recalculate ranking and priority tiers.
priority_profile["Priority_percentile"] = (
    priority_profile["Testing_priority_score"]
    .rank(method="average", pct=True)
    * 100
)

priority_profile["Priority_rank"] = (
    priority_profile["Testing_priority_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

priority_profile["Priority_tier"] = np.select(
    [
        priority_profile["Priority_percentile"] >= 90,
        priority_profile["Priority_percentile"] >= 75
    ],
    [
        "P1 — Highest 10%",
        "P2 — Next 15%"
    ],
    default="P3 — Remaining locations"
)

print("\nCORRECTED PRIORITY DISTRIBUTION")
print("=" * 65)
display(
    priority_profile["Priority_tier"]
    .value_counts()
    .to_frame("Locations")
)

print("\nCORRECTED NATIONAL SEVERITY RATES")
display(
    pd.Series(
        global_severity_rates,
        name="Corrected national severity"
    ).to_frame().round(5)
)

SULPHATE CONDITIONAL-RULE CORRECTION
------------------------------------------------------------
Conditionally unresolved observations: 5
Sulphate observations included in severity: 126321

CORRECTED PRIORITY DISTRIBUTION


,Locations
Priority_tier,
P3 — Remaining locations,18149
P2 — Next 15%,3629
P1 — Highest 10%,2421



CORRECTED NATIONAL SEVERITY RATES


,Corrected national severity
pH,0.06391
Chloride,0.10728
Magnesium,0.27139
Calcium,0.14372
Hardness,0.35796
Sulphate,0.10518
Alkalinity,0.28052
TDS,0.24883


In [10]:
top_priority = (
    priority_profile
    .sort_values("Testing_priority_score", ascending=False)
    .head(20)
)

display(
    top_priority[
        [
            "State",
            "District",
            "Latitude",
            "Longitude",
            "Testing_priority_score",
            "Priority_rank",
            "Priority_tier",
            "Chemical_component_score",
            "Temporal_component_score",
            "Monitoring_gap_score",
            "Uncertainty_component_score",
            "Chemical_parameters_tested",
            "Chemical_valid_observations",
            "Years_since_last_sample",
            "Any_sparse_trace_alert",
            "Sulphate_conditional_unresolved_records",
            "Priority_explanation"
        ]
    ].round(3)
)

# Examine the highest-priority tier.
p1_locations = priority_profile[
    priority_profile["Priority_tier"] == "P1 — Highest 10%"
]

print("\nP1 EVIDENCE COVERAGE AUDIT")
print("=" * 65)

print("P1 locations:", len(p1_locations))
print(
    "No primary chemical measurements:",
    int((p1_locations["Chemical_parameters_tested"] == 0).sum())
)
print(
    "One or two primary parameters tested:",
    int(p1_locations["Chemical_parameters_tested"].between(1, 2).sum())
)
print(
    "Three or more parameters tested:",
    int((p1_locations["Chemical_parameters_tested"] >= 3).sum())
)
print(
    "P1 with sparse trace alerts:",
    int(p1_locations["Any_sparse_trace_alert"].sum())
)
print(
    "P1 with increasing trends:",
    int((p1_locations["Significant_increasing_trend_count"] > 0).sum())
)

print("\nTOP PRIORITY COMPONENT AVERAGES")
display(
    p1_locations[
        [
            "Chemical_component_score",
            "Temporal_component_score",
            "Monitoring_gap_score",
            "Uncertainty_component_score"
        ]
    ].mean().to_frame("P1 average").round(2)
)

,State,District,Latitude,Longitude,Testing_priority_score,Priority_rank,Priority_tier,Chemical_component_score,Temporal_component_score,Monitoring_gap_score,Uncertainty_component_score,Chemical_parameters_tested,Chemical_valid_observations,Years_since_last_sample,Any_sparse_trace_alert,Sulphate_conditional_unresolved_records,Priority_explanation
_spatial_group_id,,,,,,,,,,,,,,,,,
gujarat|kachchh|lilpur|site|23.51111|70.65,gujarat,kachchh,23.511,70.650,44.267,1,P1 — Highest 10%,55.032,12.5,84.983,14.219,8,102,8.498,False,0,Elevated historical chemical exceedance eviden...
odisha|malkangiri|m.v.79|site|17.985|81.595,odisha,malkangiri,17.985,81.595,44.229,2,P1 — Highest 10%,31.885,0.0,100.000,76.500,2,2,26.497,False,0,No sample recorded in at least ten years; High...
maharashtra|jalgaon|sadavan bk.|site|20.85278|75.1,maharashtra,jalgaon,20.853,75.100,43.369,3,P1 — Highest 10%,50.766,0.0,100.000,20.417,7,51,12.498,False,0,Elevated historical chemical exceedance eviden...
gujarat|kachchh|gagodar|site|23.41667|70.8,gujarat,kachchh,23.417,70.800,43.274,4,P1 — Highest 10%,52.717,0.0,100.000,14.583,7,76,11.499,False,0,Elevated historical chemical exceedance eviden...
delhi|south west|dwarka sect 12 deep|site|28.59722|77.04639,delhi,south west,28.597,77.046,43.030,5,P1 — Highest 10%,25.606,0.0,100.000,85.250,1,1,15.499,False,0,No sample recorded in at least ten years; High...
karnataka|davanagere|jagalur|site|14.58333|76.41667,karnataka,davanagere,14.583,76.417,43.030,6,P1 — Highest 10%,25.606,0.0,100.000,85.250,1,1,22.497,False,0,No sample recorded in at least ten years; High...
gujarat|junagadh|maktupur1|site|21.14306|70.09583,gujarat,junagadh,21.143,70.096,42.804,7,P1 — Highest 10%,50.448,0.0,100.000,17.500,8,78,10.497,False,0,Elevated historical chemical exceedance eviden...
karnataka|gadag|mallapur|site|15.675|75.63333,karnataka,gadag,15.675,75.633,42.786,8,P1 — Highest 10%,25.606,0.0,100.000,83.625,1,1,23.499,False,0,No sample recorded in at least ten years; High...
andhra pradesh|prakasam|parchuru|site|15.96667|80.275,andhra pradesh,prakasam,15.967,80.275,42.716,9,P1 — Highest 10%,46.010,0.0,100.000,28.750,7,33,14.497,False,0,No sample recorded in at least ten years



P1 EVIDENCE COVERAGE AUDIT
P1 locations: 2421
No primary chemical measurements: 361
One or two primary parameters tested: 312
Three or more parameters tested: 1748
P1 with sparse trace alerts: 0
P1 with increasing trends: 10

TOP PRIORITY COMPONENT AVERAGES


,P1 average
Chemical_component_score,23.07
Temporal_component_score,0.08
Monitoring_gap_score,98.83
Uncertainty_component_score,53.33


In [11]:
PARAMETER_STALE_YEARS = 5
parameter_profile_frames = []

SITE_DISPLAY_COLUMNS = [
    "State",
    "District",
    "Latitude",
    "Longitude",
    "Priority_rank",
    "Priority_tier",
    "Testing_priority_score"
]

for spec in PRIORITY_CHEMICAL_SPECS:

    key = spec["key"]
    parameter = spec["parameter"]
    severity_col = f"_severity__{key}"

    values = pd.to_numeric(
        priority_evidence[parameter],
        errors="coerce"
    )

    valid = (
        values.notna()
        & np.isfinite(values)
        & (values >= 0)
    )

    if key == "pH":
        valid &= (values > 0) & (values <= 14)

    working = priority_evidence.loc[
        valid & priority_evidence["_date"].notna(),
        [
            "_spatial_group_id",
            "_date",
            severity_col
        ]
    ].copy()

    working["_value"] = values.loc[working.index]
    working["_year"] = working["_date"].dt.year

    working["_historical_exceedance"] = (
        working[severity_col].fillna(0) > 0
    )

    working["_severe_exceedance"] = (
        working[severity_col] >= 1
    )

    if key == "Sulphate":
        working["_conditional_unknown"] = (
            priority_evidence.loc[
                working.index,
                "_conditional_unknown__Sulphate"
            ].fillna(False).astype(bool)
        )
    else:
        working["_conditional_unknown"] = False

    grouped = (
        working.groupby("_spatial_group_id")
        .agg(
            Valid_measurements=("_value", "size"),
            Distinct_measurement_years=("_year", "nunique"),
            Latest_measurement=("_date", "max"),
            Historical_exceedance_count=(
                "_historical_exceedance", "sum"
            ),
            Historical_severe_count=(
                "_severe_exceedance", "sum"
            ),
            Conditional_unresolved_count=(
                "_conditional_unknown", "sum"
            ),
            Mean_historical_severity=(severity_col, "mean")
        )
    )

    # Include even locations with no valid historical measurement.
    parameter_profile = (
        priority_profile[SITE_DISPLAY_COLUMNS]
        .join(grouped)
    )

    parameter_profile = parameter_profile.reset_index()

    parameter_profile["Parameter"] = parameter
    parameter_profile["Parameter_key"] = key

    parameter_profile["Valid_measurements"] = (
        parameter_profile["Valid_measurements"]
        .fillna(0).astype(int)
    )

    parameter_profile["Distinct_measurement_years"] = (
        parameter_profile["Distinct_measurement_years"]
        .fillna(0).astype(int)
    )

    for col in [
        "Historical_exceedance_count",
        "Historical_severe_count",
        "Conditional_unresolved_count"
    ]:
        parameter_profile[col] = (
            parameter_profile[col].fillna(0).astype(int)
        )

    parameter_profile["Mean_historical_severity"] = (
        parameter_profile["Mean_historical_severity"]
    )

    parameter_profile["Years_since_parameter_sample"] = (
        AS_OF_DATE - parameter_profile["Latest_measurement"]
    ).dt.days / 365.25

    # Use missing status for never-measured parameters.
    parameter_profile["Has_valid_measurement"] = (
        parameter_profile["Valid_measurements"] > 0
    )

    # A site can have a physical sulphate measurement but an unresolved
    # conditional regulatory assessment. Keep these as separate statuses.
    parameter_profile_frames.append(parameter_profile)

parameter_location_profile = pd.concat(
    parameter_profile_frames,
    ignore_index=True
)

# Verify one row per location and scored parameter.
assert not parameter_location_profile.duplicated(
    ["_spatial_group_id", "Parameter"]
).any()

assert len(parameter_location_profile) == (
    len(priority_profile) * len(PRIORITY_CHEMICAL_SPECS)
)

print("PARAMETER-LEVEL MONITORING PROFILE")
print("=" * 75)
print("Provisional locations:", f"{len(priority_profile):,}")
print("Parameters:", len(PRIORITY_CHEMICAL_SPECS))
print("Location-parameter rows:", f"{len(parameter_location_profile):,}")

display(
    parameter_location_profile[
        [
            "Parameter",
            "Valid_measurements",
            "Distinct_measurement_years",
            "Historical_exceedance_count",
            "Historical_severe_count",
            "Conditional_unresolved_count",
            "Latest_measurement",
            "Years_since_parameter_sample"
        ]
    ].groupby("Parameter").agg(
        Locations_measured=("Valid_measurements", lambda x: (x > 0).sum()),
        Locations_never_measured=("Valid_measurements", lambda x: (x == 0).sum()),
        Locations_with_exceedance=("Historical_exceedance_count", lambda x: (x > 0).sum()),
        Median_years_since_sample=("Years_since_parameter_sample", "median")
    ).round(2)
)

PARAMETER-LEVEL MONITORING PROFILE
Provisional locations: 24,199
Parameters: 8
Location-parameter rows: 193,592


,Locations_measured,Locations_never_measured,Locations_with_exceedance,Median_years_since_sample
Parameter,,,,
Calcium (mg/L),23343,856,11188,6.48
Chloride (mg/L),23674,525,8035,6.48
Magnesium (mg/L),23341,858,16529,6.48
Potential of Hydrogen (pH),23692,507,6083,6.48
Sulphate (mg/L),21967,2232,4864,8.50
Total Alkalinity (mg/L as CaCO3),17765,6434,11751,9.50
Total Dissolved Solids (mg/L),14631,9568,8019,7.48
Total Hardness (mgCaCO3/L),23414,785,18566,7.48


In [12]:
# Build a lookup for significant increasing May-only trends.
may_increasing_lookup = {}

for target in PRIORITY_TARGETS:

    subset = may_priority_trends[
        (may_priority_trends["Target"] == target)
        & (may_priority_trends["BH_q_value"] < 0.05)
        & (may_priority_trends["Theil_Sen_slope_per_year"] > 0)
    ]

    may_increasing_lookup[target] = set(
        subset["_spatial_group_id"].astype(str)
    )

parameter_location_profile[
    "Significant_increasing_trend"
] = False

for target in PRIORITY_TARGETS:

    parameter_mask = (
        parameter_location_profile["Parameter"] == target
    )

    parameter_location_profile.loc[
        parameter_mask,
        "Significant_increasing_trend"
    ] = (
        parameter_location_profile.loc[
            parameter_mask, "_spatial_group_id"
        ].astype(str).isin(may_increasing_lookup[target]).to_numpy()
    )

def build_parameter_recommendation(row):

    reasons = []

    if not row["Has_valid_measurement"]:
        reasons.append("No valid historical measurement; establish baseline")

    if row["Historical_exceedance_count"] > 0:
        reasons.append(
            f"{int(row['Historical_exceedance_count'])} historical exceedance record(s)"
        )

    if row["Historical_severe_count"] > 0:
        reasons.append(
            f"{int(row['Historical_severe_count'])} severe exceedance record(s)"
        )

    if row["Significant_increasing_trend"]:
        reasons.append("Significant increasing May-only trend")

    if row["Years_since_parameter_sample"] >= PARAMETER_STALE_YEARS:
        reasons.append(
            f"Parameter not measured for at least {PARAMETER_STALE_YEARS} years"
        )

    if row["Conditional_unresolved_count"] > 0:
        reasons.append(
            "Conditional sulphate assessment needs magnesium-rule review"
        )

    if not reasons:
        reasons.append("No specific retest trigger identified")

    return "; ".join(reasons)


parameter_location_profile["Testing_reason"] = (
    parameter_location_profile.apply(
        build_parameter_recommendation,
        axis=1
    )
)

def classify_parameter_recommendation(row):

    if not row["Has_valid_measurement"]:
        return "Baseline test required"

    if (
        row["Historical_exceedance_count"] > 0
        or row["Significant_increasing_trend"]
        or row["Conditional_unresolved_count"] > 0
    ):
        return "Targeted retest"

    if row["Years_since_parameter_sample"] >= 10:
        return "Overdue retest"

    if row["Years_since_parameter_sample"] >= PARAMETER_STALE_YEARS:
        return "Routine retest due"

    return "Routine monitoring"


parameter_location_profile["Testing_recommendation"] = (
    parameter_location_profile.apply(
        classify_parameter_recommendation,
        axis=1
    )
)

print("PARAMETER-LEVEL TESTING RECOMMENDATIONS")
print("=" * 75)

recommendation_summary = (
    parameter_location_profile
    .groupby(["Parameter", "Testing_recommendation"])
    .size()
    .unstack(fill_value=0)
)

display(recommendation_summary)

print("\nOVERALL RECOMMENDATION DISTRIBUTION")
display(
    parameter_location_profile["Testing_recommendation"]
    .value_counts()
    .to_frame("Location-parameter pairs")
)

PARAMETER-LEVEL TESTING RECOMMENDATIONS


Testing_recommendation,Baseline test required,Overdue retest,Routine monitoring,Routine retest due,Targeted retest
Parameter,,,,,
Calcium (mg/L),856,5118,2208,4828,11189
Chloride (mg/L),525,5581,3728,6297,8068
Magnesium (mg/L),858,2753,1248,2811,16529
Potential of Hydrogen (pH),507,6155,3956,7498,6083
Sulphate (mg/L),2232,6260,4246,6597,4864
Total Alkalinity (mg/L as CaCO3),6434,2798,382,2834,11751
Total Dissolved Solids (mg/L),9568,507,1296,4809,8019
Total Hardness (mgCaCO3/L),785,2159,775,1914,18566



OVERALL RECOMMENDATION DISTRIBUTION


,Location-parameter pairs
Testing_recommendation,
Targeted retest,85069
Routine retest due,37588
Overdue retest,31331
Baseline test required,21765
Routine monitoring,17839


In [13]:
# Select P1 locations and their parameter-level recommendations.
p1_parameter_plan = parameter_location_profile[
    parameter_location_profile["Priority_tier"] == "P1 — Highest 10%"
].copy()

# Separate records needing an action from routine surveillance.
action_recommendations = p1_parameter_plan[
    p1_parameter_plan["Testing_recommendation"].isin([
        "Baseline test required",
        "Targeted retest",
        "Overdue retest",
        "Routine retest due"
    ])
].copy()

# Consolidate proposed analytes by monitoring location.
def join_parameters(values):
    return ", ".join(sorted(set(values)))

p1_testing_plan = (
    action_recommendations
    .groupby("_spatial_group_id")
    .agg(
        State=("State", "first"),
        District=("District", "first"),
        Latitude=("Latitude", "first"),
        Longitude=("Longitude", "first"),
        Priority_rank=("Priority_rank", "first"),
        Testing_priority_score=("Testing_priority_score", "first"),
        Parameter_tests_recommended=("Parameter", join_parameters),
        Number_of_parameter_tests=("Parameter", "nunique"),
        Baseline_tests=("Testing_recommendation",
                        lambda x: (x == "Baseline test required").sum()),
        Targeted_retests=("Testing_recommendation",
                          lambda x: (x == "Targeted retest").sum()),
        Overdue_retests=("Testing_recommendation",
                        lambda x: (x == "Overdue retest").sum()),
        Routine_retests=("Testing_recommendation",
                         lambda x: (x == "Routine retest due").sum())
    )
    .sort_values("Priority_rank")
)

print("P1 PARAMETER-SPECIFIC TESTING PLAN")
print("=" * 85)
display(p1_testing_plan.head(20))

print("\nP1 TESTING PLAN SUMMARY")
display(
    p1_parameter_plan
    .groupby("Testing_recommendation")
    .agg(
        Location_parameter_pairs=("Parameter", "size"),
        Unique_locations=("_spatial_group_id", "nunique")
    )
)

print("\nTop P1 site count:", len(p1_testing_plan))
print(
    "Mean recommended tests per P1 site:",
    round(p1_testing_plan["Number_of_parameter_tests"].mean(), 2)
)

P1 PARAMETER-SPECIFIC TESTING PLAN


,State,District,Latitude,Longitude,Priority_rank,Testing_priority_score,Parameter_tests_recommended,Number_of_parameter_tests,Baseline_tests,Targeted_retests,Overdue_retests,Routine_retests
_spatial_group_id,,,,,,,,,,,,
gujarat|kachchh|lilpur|site|23.51111|70.65,gujarat,kachchh,23.511111,70.650000,1,44.267053,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,0,7,0,1
odisha|malkangiri|m.v.79|site|17.985|81.595,odisha,malkangiri,17.985000,81.595000,2,44.229041,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,6,2,0,0
maharashtra|jalgaon|sadavan bk.|site|20.85278|75.1,maharashtra,jalgaon,20.852778,75.100000,3,43.369090,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,1,6,1,0
gujarat|kachchh|gagodar|site|23.41667|70.8,gujarat,kachchh,23.416667,70.800000,4,43.274393,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,1,6,1,0
delhi|south west|dwarka sect 12 deep|site|28.59722|77.04639,delhi,south west,28.597222,77.046389,5,43.030084,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,7,1,0,0
karnataka|davanagere|jagalur|site|14.58333|76.41667,karnataka,davanagere,14.583333,76.416667,6,43.030084,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,7,1,0,0
gujarat|junagadh|maktupur1|site|21.14306|70.09583,gujarat,junagadh,21.143056,70.095833,7,42.804128,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,0,7,1,0
karnataka|gadag|mallapur|site|15.675|75.63333,karnataka,gadag,15.675000,75.633333,8,42.786334,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,7,1,0,0
andhra pradesh|prakasam|parchuru|site|15.96667|80.275,andhra pradesh,prakasam,15.966667,80.275000,9,42.716397,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,1,7,0,0



P1 TESTING PLAN SUMMARY


,Location_parameter_pairs,Unique_locations
Testing_recommendation,,
Baseline test required,7385,2211
Overdue retest,4331,1716
Routine retest due,195,152
Targeted retest,7457,1746



Top P1 site count: 2421
Mean recommended tests per P1 site: 8.0


In [14]:
exceedance_history_frames = []

for spec in PRIORITY_CHEMICAL_SPECS:

    key = spec["key"]
    parameter = spec["parameter"]
    severity_col = f"_severity__{key}"

    events = priority_evidence.loc[
        priority_evidence[severity_col].gt(0)
        & priority_evidence["_date"].notna(),
        [
            "_spatial_group_id",
            "_date",
            severity_col
        ]
    ].copy()

    events["_year"] = events["_date"].dt.year

    event_stats = (
        events.groupby("_spatial_group_id")
        .agg(
            Exceedance_count=(severity_col, "count"),
            Severe_exceedance_count=(
                severity_col,
                lambda x: (x >= 1).sum()
            ),
            Distinct_exceedance_years=("_year", "nunique"),
            Latest_exceedance_date=("_date", "max")
        )
    )

    event_stats = event_stats.reset_index()
    event_stats["Parameter"] = parameter

    exceedance_history_frames.append(event_stats)

exceedance_history_df = pd.concat(
    exceedance_history_frames,
    ignore_index=True
)

# Merge onto the complete parameter-location register.
parameter_location_audit = parameter_location_profile.merge(
    exceedance_history_df,
    on=["_spatial_group_id", "Parameter"],
    how="left",
    validate="one_to_one"
)

for col in [
    "Exceedance_count",
    "Severe_exceedance_count",
    "Distinct_exceedance_years"
]:
    parameter_location_audit[col] = (
        parameter_location_audit[col].fillna(0).astype(int)
    )

parameter_location_audit["Years_since_last_exceedance"] = (
    AS_OF_DATE
    - parameter_location_audit["Latest_exceedance_date"]
).dt.days / 365.25

parameter_location_audit["Exceedance_recency"] = np.select(
    [
        parameter_location_audit["Latest_exceedance_date"].isna(),
        parameter_location_audit["Years_since_last_exceedance"] <= 5,
        parameter_location_audit["Years_since_last_exceedance"] <= 10
    ],
    [
        "No historical exceedance",
        "Recent — within 5 years",
        "Intermediate — 5 to 10 years"
    ],
    default="Old — more than 10 years"
)

print("HISTORICAL EXCEEDANCE RECENCY AUDIT")
print("=" * 80)

p1_audit = parameter_location_audit[
    parameter_location_audit["Priority_tier"] == "P1 — Highest 10%"
].copy()

p1_exceedance_summary = (
    p1_audit.groupby("Parameter")
    .agg(
        P1_locations=("_spatial_group_id", "nunique"),
        Historical_exceedance=(
            "Exceedance_count", lambda x: (x > 0).sum()
        ),
        Recent_exceedance=(
            "Years_since_last_exceedance",
            lambda x: (x <= 5).sum()
        ),
        Intermediate_exceedance=(
            "Years_since_last_exceedance",
            lambda x: ((x > 5) & (x <= 10)).sum()
        ),
        Old_exceedance=(
            "Years_since_last_exceedance",
            lambda x: (x > 10).sum()
        ),
        Repeated_exceedance=(
            "Distinct_exceedance_years",
            lambda x: (x >= 2).sum()
        ),
        Severe_exceedance=(
            "Severe_exceedance_count",
            lambda x: (x > 0).sum()
        )
    )
)

display(p1_exceedance_summary)

print("\nP1 LOCATION-PARAMETER RECENCY DISTRIBUTION")
display(
    p1_audit.groupby(
        ["Parameter", "Exceedance_recency"]
    ).size().unstack(fill_value=0)
)

print("\nP1 LOCATION-PARAMETER PAIRS:", len(p1_audit))
print(
    "Pairs with no historical exceedance:",
    int((p1_audit["Exceedance_count"] == 0).sum())
)

HISTORICAL EXCEEDANCE RECENCY AUDIT


,P1_locations,Historical_exceedance,Recent_exceedance,Intermediate_exceedance,Old_exceedance,Repeated_exceedance,Severe_exceedance
Parameter,,,,,,,
Calcium (mg/L),2421,927,0,122,805,511,380
Chloride (mg/L),2421,1155,0,170,985,740,459
Magnesium (mg/L),2421,1413,0,144,1269,766,723
Potential of Hydrogen (pH),2421,510,0,37,473,154,510
Sulphate (mg/L),2421,823,0,113,710,476,816
Total Alkalinity (mg/L as CaCO3),2421,939,0,78,861,503,210
Total Dissolved Solids (mg/L),2421,198,0,84,114,55,133
Total Hardness (mgCaCO3/L),2421,1492,0,148,1344,786,802



P1 LOCATION-PARAMETER RECENCY DISTRIBUTION


Exceedance_recency,Intermediate — 5 to 10 years,No historical exceedance,Old — more than 10 years
Parameter,,,
Calcium (mg/L),122,1494,805
Chloride (mg/L),170,1266,985
Magnesium (mg/L),144,1008,1269
Potential of Hydrogen (pH),37,1911,473
Sulphate (mg/L),113,1598,710
Total Alkalinity (mg/L as CaCO3),78,1482,861
Total Dissolved Solids (mg/L),84,2223,114
Total Hardness (mgCaCO3/L),148,929,1344



P1 LOCATION-PARAMETER PAIRS: 19368
Pairs with no historical exceedance: 11911


In [15]:
# Make sure the increasing-trend indicator is available.
trend_flags = parameter_location_profile[
    [
        "_spatial_group_id",
        "Parameter",
        "Significant_increasing_trend"
    ]
].copy()

parameter_location_audit = parameter_location_audit.drop(
    columns=["Significant_increasing_trend"],
    errors="ignore"
).merge(
    trend_flags,
    on=["_spatial_group_id", "Parameter"],
    how="left",
    validate="one_to_one"
)

parameter_location_audit[
    "Significant_increasing_trend"
] = (
    parameter_location_audit[
        "Significant_increasing_trend"
    ].fillna(False).astype(bool)
)


def revised_testing_recommendation(row):

    age = row["Years_since_parameter_sample"]
    exceedance_age = row["Years_since_last_exceedance"]

    has_measurement = row["Has_valid_measurement"]

    severe = row["Severe_exceedance_count"] > 0
    repeated = row["Distinct_exceedance_years"] >= 2

    recent_exceedance = (
        pd.notna(exceedance_age)
        and exceedance_age <= 5
    )

    intermediate_exceedance = (
        pd.notna(exceedance_age)
        and 5 < exceedance_age <= 10
        and (severe or repeated)
    )

    increasing_trend = (
        row["Significant_increasing_trend"]
        and pd.notna(age)
        and age <= 5
    )

    conditional_review = (
        row["Conditional_unresolved_count"] > 0
        and pd.notna(age)
        and age <= 10
    )

    reasons = []

    # First priority: establish missing baseline measurements.
    if not has_measurement:
        return (
            "Baseline test required",
            "No valid historical measurement for this parameter"
        )

    # Targeted testing requires more recent or repeated evidence.
    if recent_exceedance:
        reasons.append("Historical exceedance within five years")

    if intermediate_exceedance:
        reasons.append(
            "Severe or repeated exceedance within 5–10 years"
        )

    if increasing_trend:
        reasons.append(
            "Significant increasing trend with recent monitoring"
        )

    if conditional_review:
        reasons.append(
            "Conditional sulphate assessment requires clarification"
        )

    if reasons:
        return "Targeted retest", "; ".join(reasons)

    # Monitoring gaps take precedence over older exceedance history.
    if pd.notna(age) and age >= 10:
        return (
            "Overdue retest",
            "Latest valid parameter measurement is at least ten years old"
        )

    if pd.notna(age) and age >= 5:
        return (
            "Routine retest due",
            "Latest valid parameter measurement is 5–10 years old"
        )

    return (
        "Routine monitoring",
        "No immediate parameter-specific retest trigger identified"
    )


recommendation_results = parameter_location_audit.apply(
    revised_testing_recommendation,
    axis=1
)

parameter_location_audit["Testing_recommendation"] = [
    result[0] for result in recommendation_results
]

parameter_location_audit["Testing_reason"] = [
    result[1] for result in recommendation_results
]

print("REVISED PARAMETER-LEVEL RECOMMENDATIONS")
print("=" * 75)

display(
    parameter_location_audit
    .groupby(["Parameter", "Testing_recommendation"])
    .size()
    .unstack(fill_value=0)
)

print("\nOVERALL RECOMMENDATION DISTRIBUTION")

display(
    parameter_location_audit[
        "Testing_recommendation"
    ].value_counts().to_frame("Location-parameter pairs")
)

REVISED PARAMETER-LEVEL RECOMMENDATIONS


Testing_recommendation,Baseline test required,Overdue retest,Routine monitoring,Routine retest due,Targeted retest
Parameter,,,,,
Calcium (mg/L),856,7643,3222,7017,5461
Chloride (mg/L),525,7656,4545,8072,3401
Magnesium (mg/L),858,7645,2068,4940,8688
Potential of Hydrogen (pH),507,7682,4893,8985,2132
Sulphate (mg/L),2232,7578,4924,7474,1991
Total Alkalinity (mg/L as CaCO3),6434,8211,439,6122,2993
Total Dissolved Solids (mg/L),9568,987,1641,7308,4695
Total Hardness (mgCaCO3/L),785,7733,1310,4374,9997



OVERALL RECOMMENDATION DISTRIBUTION


,Location-parameter pairs
Testing_recommendation,
Overdue retest,55135
Routine retest due,54292
Targeted retest,39358
Routine monitoring,23042
Baseline test required,21765


In [16]:
p1_parameter_plan = parameter_location_audit[
    parameter_location_audit["Priority_tier"] == "P1 — Highest 10%"
].copy()

action_recommendations = p1_parameter_plan[
    p1_parameter_plan["Testing_recommendation"].isin([
        "Baseline test required",
        "Targeted retest",
        "Overdue retest",
        "Routine retest due"
    ])
].copy()

def join_parameters(values):
    return ", ".join(sorted(set(values)))


p1_testing_plan = (
    action_recommendations
    .groupby("_spatial_group_id")
    .agg(
        State=("State", "first"),
        District=("District", "first"),
        Latitude=("Latitude", "first"),
        Longitude=("Longitude", "first"),
        Priority_rank=("Priority_rank", "first"),
        Testing_priority_score=("Testing_priority_score", "first"),
        Recommended_parameters=("Parameter", join_parameters),
        Number_of_tests=("Parameter", "nunique"),
        Baseline_tests=(
            "Testing_recommendation",
            lambda x: (x == "Baseline test required").sum()
        ),
        Targeted_retests=(
            "Testing_recommendation",
            lambda x: (x == "Targeted retest").sum()
        ),
        Overdue_retests=(
            "Testing_recommendation",
            lambda x: (x == "Overdue retest").sum()
        ),
        Routine_retests=(
            "Testing_recommendation",
            lambda x: (x == "Routine retest due").sum()
        )
    )
    .sort_values("Priority_rank")
)

print("REVISED P1 TESTING PLAN")
print("=" * 85)
display(p1_testing_plan.head(20))

print("\nREVISED P1 RECOMMENDATION DISTRIBUTION")
display(
    p1_parameter_plan.groupby("Testing_recommendation")
    .agg(
        Location_parameter_pairs=("Parameter", "size"),
        Unique_locations=("_spatial_group_id", "nunique")
    )
)

print("\nTESTING PLAN SUMMARY")
print("P1 locations:", len(p1_testing_plan))
print(
    "Average recommended tests per P1 location:",
    round(p1_testing_plan["Number_of_tests"].mean(), 2)
)
print(
    "Total proposed P1 parameter tests:",
    int(p1_testing_plan["Number_of_tests"].sum())
)

REVISED P1 TESTING PLAN


,State,District,Latitude,Longitude,Priority_rank,Testing_priority_score,Recommended_parameters,Number_of_tests,Baseline_tests,Targeted_retests,Overdue_retests,Routine_retests
_spatial_group_id,,,,,,,,,,,,
gujarat|kachchh|lilpur|site|23.51111|70.65,gujarat,kachchh,23.511111,70.650000,1,44.267053,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,0,7,0,1
odisha|malkangiri|m.v.79|site|17.985|81.595,odisha,malkangiri,17.985000,81.595000,2,44.229041,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,6,0,2,0
maharashtra|jalgaon|sadavan bk.|site|20.85278|75.1,maharashtra,jalgaon,20.852778,75.100000,3,43.369090,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,1,0,7,0
gujarat|kachchh|gagodar|site|23.41667|70.8,gujarat,kachchh,23.416667,70.800000,4,43.274393,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,1,0,7,0
delhi|south west|dwarka sect 12 deep|site|28.59722|77.04639,delhi,south west,28.597222,77.046389,5,43.030084,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,7,0,1,0
karnataka|davanagere|jagalur|site|14.58333|76.41667,karnataka,davanagere,14.583333,76.416667,6,43.030084,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,7,0,1,0
gujarat|junagadh|maktupur1|site|21.14306|70.09583,gujarat,junagadh,21.143056,70.095833,7,42.804128,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,0,0,8,0
karnataka|gadag|mallapur|site|15.675|75.63333,karnataka,gadag,15.675000,75.633333,8,42.786334,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,7,0,1,0
andhra pradesh|prakasam|parchuru|site|15.96667|80.275,andhra pradesh,prakasam,15.966667,80.275000,9,42.716397,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",8,1,0,7,0



REVISED P1 RECOMMENDATION DISTRIBUTION


,Location_parameter_pairs,Unique_locations
Testing_recommendation,,
Baseline test required,7385,2211
Overdue retest,10663,1930
Routine retest due,462,213
Targeted retest,858,195



TESTING PLAN SUMMARY
P1 locations: 2421
Average recommended tests per P1 location: 8.0
Total proposed P1 parameter tests: 19368


In [17]:
# Verify missing measurements always receive baseline recommendations.
missing_measurement_rows = parameter_location_audit[
    ~parameter_location_audit["Has_valid_measurement"]
]

assert (
    missing_measurement_rows["Testing_recommendation"]
    == "Baseline test required"
).all()

# Verify no old exceedance alone triggers a targeted retest.
old_only = parameter_location_audit[
    parameter_location_audit["Has_valid_measurement"]
    & (parameter_location_audit["Years_since_last_exceedance"] > 10)
    & ~parameter_location_audit["Significant_increasing_trend"]
]

# Conditional assessment triggers are allowed only within ten years.
old_only_no_conditional = old_only[
    old_only["Conditional_unresolved_count"] == 0
]

assert not (
    old_only_no_conditional["Testing_recommendation"]
    == "Targeted retest"
).any()

# Verify every location has one recommendation per scored parameter.
assert not parameter_location_audit.duplicated(
    ["_spatial_group_id", "Parameter"]
).any()

print("RECOMMENDATION LOGIC VALIDATION")
print("=" * 65)
print("Missing parameters receive baseline recommendations: PASSED")
print("Old exceedances alone do not trigger targeted retests: PASSED")
print("Location-parameter uniqueness: PASSED")
print("Total location-parameter rows:", len(parameter_location_audit))

print("\nP1 targeted retest pairs:",
      int((p1_parameter_plan["Testing_recommendation"] == "Targeted retest").sum()))

RECOMMENDATION LOGIC VALIDATION
Missing parameters receive baseline recommendations: PASSED
Old exceedances alone do not trigger targeted retests: PASSED
Location-parameter uniqueness: PASSED
Total location-parameter rows: 193592

P1 targeted retest pairs: 858


In [18]:
# Exclude routine monitoring from the immediate action list.
action_rows = parameter_location_audit[
    parameter_location_audit["Testing_recommendation"] != "Routine monitoring"
].copy()


def parameters_for_action(group, action):
    return ", ".join(sorted(
        group.loc[
            group["Testing_recommendation"] == action,
            "Parameter"
        ].unique()
    ))


def action_reasons(group):
    return " | ".join(
        f"{row.Parameter}: {row.Testing_reason}"
        for row in group.itertuples()
        if row.Testing_recommendation != "Routine monitoring"
    )


site_actions = (
    action_rows.groupby("_spatial_group_id")
    .apply(
        lambda g: pd.Series({
            "Recommended_test_count": g["Parameter"].nunique(),

            "Baseline_test_count": (
                g["Testing_recommendation"] == "Baseline test required"
            ).sum(),

            "Targeted_retest_count": (
                g["Testing_recommendation"] == "Targeted retest"
            ).sum(),

            "Overdue_retest_count": (
                g["Testing_recommendation"] == "Overdue retest"
            ).sum(),

            "Routine_retest_due_count": (
                g["Testing_recommendation"] == "Routine retest due"
            ).sum(),

            "Baseline_parameters": parameters_for_action(
                g, "Baseline test required"
            ),

            "Targeted_parameters": parameters_for_action(
                g, "Targeted retest"
            ),

            "Overdue_parameters": parameters_for_action(
                g, "Overdue retest"
            ),

            "Routine_retest_parameters": parameters_for_action(
                g, "Routine retest due"
            ),

            "Action_reason_summary": action_reasons(g)
        }),
        include_groups=False
    )
)

# Keep all locations, including those with no immediate testing action.
site_action_register = priority_profile.join(
    site_actions,
    how="left"
)

action_count_columns = [
    "Recommended_test_count",
    "Baseline_test_count",
    "Targeted_retest_count",
    "Overdue_retest_count",
    "Routine_retest_due_count"
]

for col in action_count_columns:
    site_action_register[col] = (
        site_action_register[col].fillna(0).astype(int)
    )

for col in [
    "Baseline_parameters",
    "Targeted_parameters",
    "Overdue_parameters",
    "Routine_retest_parameters",
    "Action_reason_summary"
]:
    site_action_register[col] = (
        site_action_register[col].fillna("")
    )

# Make sparse historical trace alerts explicit.
trace_parameter_names = list(TRACE_ALERT_SPECS)

trace_alert_columns = [
    f"Trace_alerts__{key}"
    for key in trace_parameter_names
]

site_action_register["Sparse_trace_alert_parameters"] = (
    site_action_register.apply(
        lambda row: ", ".join(
            key for key in trace_parameter_names
            if row[f"Trace_alerts__{key}"] > 0
        ),
        axis=1
    )
)

print("CONSOLIDATED LOCATION ACTION REGISTER")
print("=" * 75)

print("Total locations:", f"{len(site_action_register):,}")
print(
    "Locations with one or more recommended tests:",
    int((site_action_register["Recommended_test_count"] > 0).sum())
)

print("\nP1 ACTION REGISTER — FIRST 20 LOCATIONS")

display(
    site_action_register[
        site_action_register["Priority_tier"] == "P1 — Highest 10%"
    ]
    .sort_values("Priority_rank")
    [
        [
            "State",
            "District",
            "Priority_rank",
            "Testing_priority_score",
            "Recommended_test_count",
            "Baseline_test_count",
            "Targeted_retest_count",
            "Overdue_retest_count",
            "Routine_retest_due_count",
            "Targeted_parameters",
            "Baseline_parameters",
            "Sparse_trace_alert_parameters"
        ]
    ].head(20).round(3)
)

print("\nALL-LOCATION TESTING ACTION SUMMARY")

display(
    site_action_register.groupby("Priority_tier")
    .agg(
        Locations=("_record_id" if "_record_id" in priority_profile.columns else
                   "Total_records", "size"),
        Mean_recommended_tests=("Recommended_test_count", "mean"),
        Locations_with_targeted_retest=(
            "Targeted_retest_count", lambda x: (x > 0).sum()
        ),
        Locations_with_baseline=(
            "Baseline_test_count", lambda x: (x > 0).sum()
        )
    ).round(2)
)

CONSOLIDATED LOCATION ACTION REGISTER
Total locations: 24,199
Locations with one or more recommended tests: 24008

P1 ACTION REGISTER — FIRST 20 LOCATIONS


,State,District,Priority_rank,Testing_priority_score,Recommended_test_count,Baseline_test_count,Targeted_retest_count,Overdue_retest_count,Routine_retest_due_count,Targeted_parameters,Baseline_parameters,Sparse_trace_alert_parameters
_spatial_group_id,,,,,,,,,,,,
gujarat|kachchh|lilpur|site|23.51111|70.65,gujarat,kachchh,1,44.267,8,0,7,0,1,"Calcium (mg/L), Chloride (mg/L), Magnesium (mg...",,
odisha|malkangiri|m.v.79|site|17.985|81.595,odisha,malkangiri,2,44.229,8,6,0,2,0,,"Calcium (mg/L), Magnesium (mg/L), Potential of...",
maharashtra|jalgaon|sadavan bk.|site|20.85278|75.1,maharashtra,jalgaon,3,43.369,8,1,0,7,0,,Total Dissolved Solids (mg/L),
gujarat|kachchh|gagodar|site|23.41667|70.8,gujarat,kachchh,4,43.274,8,1,0,7,0,,Total Dissolved Solids (mg/L),
delhi|south west|dwarka sect 12 deep|site|28.59722|77.04639,delhi,south west,5,43.030,8,7,0,1,0,,"Calcium (mg/L), Magnesium (mg/L), Potential of...",
karnataka|davanagere|jagalur|site|14.58333|76.41667,karnataka,davanagere,6,43.030,8,7,0,1,0,,"Calcium (mg/L), Magnesium (mg/L), Potential of...",
gujarat|junagadh|maktupur1|site|21.14306|70.09583,gujarat,junagadh,7,42.804,8,0,0,8,0,,,
karnataka|gadag|mallapur|site|15.675|75.63333,karnataka,gadag,8,42.786,8,7,0,1,0,,"Calcium (mg/L), Magnesium (mg/L), Potential of...",
andhra pradesh|prakasam|parchuru|site|15.96667|80.275,andhra pradesh,prakasam,9,42.716,8,1,0,7,0,,Total Dissolved Solids (mg/L),



ALL-LOCATION TESTING ACTION SUMMARY


,Locations,Mean_recommended_tests,Locations_with_targeted_retest,Locations_with_baseline
Priority_tier,,,,
P1 — Highest 10%,2421,8.00,195,2211
P2 — Next 15%,3629,8.00,493,3142
P3 — Remaining locations,18149,6.73,11363,6841


In [20]:
from scipy.stats import spearmanr

SCORE_COMPONENTS = {
    "Chemical": "Chemical_component_score",
    "Temporal": "Temporal_component_score",
    "Monitoring_gap": "Monitoring_gap_score",
    "Uncertainty": "Uncertainty_component_score"
}

WEIGHT_SCENARIOS = {
    "Current_balanced": {
        "Chemical": 0.40,
        "Temporal": 0.25,
        "Monitoring_gap": 0.20,
        "Uncertainty": 0.15
    },
    "Chemical_emphasis": {
        "Chemical": 0.50,
        "Temporal": 0.20,
        "Monitoring_gap": 0.20,
        "Uncertainty": 0.10
    },
    "Monitoring_gap_emphasis": {
        "Chemical": 0.30,
        "Temporal": 0.20,
        "Monitoring_gap": 0.35,
        "Uncertainty": 0.15
    },
    "Uncertainty_emphasis": {
        "Chemical": 0.30,
        "Temporal": 0.20,
        "Monitoring_gap": 0.20,
        "Uncertainty": 0.30
    }
}

scenario_scores = pd.DataFrame(
    index=priority_profile.index
)

for scenario_name, weights in WEIGHT_SCENARIOS.items():

    assert abs(sum(weights.values()) - 1.0) < 1e-9

    score = pd.Series(
        0.0,
        index=priority_profile.index
    )

    for component_name, weight in weights.items():

        column = SCORE_COMPONENTS[component_name]

        score += (
            priority_profile[column].fillna(0) * weight
        )

    scenario_scores[scenario_name] = score

# Compare all scenarios with the current balanced ranking.
current_rank = (
    scenario_scores["Current_balanced"]
    .rank(ascending=False, method="average")
)

top_n = int(np.ceil(len(priority_profile) * 0.10))

baseline_top = set(
    scenario_scores["Current_balanced"]
    .nlargest(top_n).index
)

sensitivity_rows = []

for scenario_name in WEIGHT_SCENARIOS:

    scenario_rank = (
        scenario_scores[scenario_name]
        .rank(ascending=False, method="average")
    )

    scenario_top = set(
        scenario_scores[scenario_name]
        .nlargest(top_n).index
    )

    rho, p_value = spearmanr(
        current_rank,
        scenario_rank
    )

    sensitivity_rows.append({
        "Scenario": scenario_name,
        "Spearman_rank_correlation": rho,
        "Top_10_percent_overlap": len(
            baseline_top & scenario_top
        ),
        "Top_10_percent_overlap_percentage": (
            len(baseline_top & scenario_top) / top_n * 100
        ),
        "Top_n_locations": top_n
    })

weight_sensitivity_df = (
    pd.DataFrame(sensitivity_rows)
    .set_index("Scenario")
)

print("PRIORITY WEIGHT SENSITIVITY")
print("=" * 75)
display(weight_sensitivity_df.round(4))

# Save scenario scores for subsequent validation.
priority_scenario_scores = scenario_scores.copy()

PRIORITY WEIGHT SENSITIVITY


,Spearman_rank_correlation,Top_10_percent_overlap,Top_10_percent_overlap_percentage,Top_n_locations
Scenario,,,,
Current_balanced,1.0000,2420,100.0000,2420
Chemical_emphasis,0.9860,1911,78.9669,2420
Monitoring_gap_emphasis,0.9759,2066,85.3719,2420
Uncertainty_emphasis,0.9663,1762,72.8099,2420


In [21]:
# Calculate each component's actual weighted contribution.
for component_name, column in SCORE_COMPONENTS.items():

    current_weight = WEIGHT_SCENARIOS[
        "Current_balanced"
    ][component_name]

    priority_profile[
        f"Weighted_contribution__{component_name}"
    ] = (
        priority_profile[column] * current_weight
    )

weighted_contribution_columns = [
    f"Weighted_contribution__{name}"
    for name in SCORE_COMPONENTS
]

contribution_to_driver = (
    priority_profile[weighted_contribution_columns]
    .idxmax(axis=1)
    .str.replace("Weighted_contribution__", "", regex=False)
)

priority_profile["Primary_priority_driver"] = (
    contribution_to_driver
)

p1_driver_summary = (
    priority_profile[
        priority_profile["Priority_tier"] == "P1 — Highest 10%"
    ]
    .groupby("Primary_priority_driver")
    .agg(
        Locations=("_spatial_group_id" if "_spatial_group_id" in priority_profile.columns
                   else "Total_records", "size"),
        Mean_priority_score=("Testing_priority_score", "mean"),
        Mean_chemical_score=("Chemical_component_score", "mean"),
        Mean_monitoring_gap_score=("Monitoring_gap_score", "mean"),
        Mean_uncertainty_score=("Uncertainty_component_score", "mean")
    )
)

print("PRIMARY DRIVER OF P1 PRIORITIES")
print("=" * 75)
display(p1_driver_summary.round(2))

print("\nP1 CHEMICAL EVIDENCE COVERAGE")
display(
    priority_profile[
        priority_profile["Priority_tier"] == "P1 — Highest 10%"
    ]
    .groupby("Chemical_parameters_tested")
    .size()
    .rename("P1 locations")
    .to_frame()
)

print("\nP1 priority score distribution")
display(
    priority_profile.loc[
        priority_profile["Priority_tier"] == "P1 — Highest 10%",
        "Testing_priority_score"
    ].describe().to_frame("P1 score").round(3)
)

PRIMARY DRIVER OF P1 PRIORITIES


,Locations,Mean_priority_score,Mean_chemical_score,Mean_monitoring_gap_score,Mean_uncertainty_score
Primary_priority_driver,,,,,
Chemical,67,37.48,47.88,79.77,13.35
Monitoring_gap,2354,37.00,22.36,99.38,54.47



P1 CHEMICAL EVIDENCE COVERAGE


,P1 locations
Chemical_parameters_tested,
0,361
1,103
2,209
3,47
4,27
5,182
6,351
7,931
8,210



P1 priority score distribution


,P1 score
count,2421.000
mean,37.013
std,2.540
min,34.339
25%,35.093
50%,36.117
75%,38.012
max,44.267


In [22]:
# CELL 18 — Separate review registers and tier-level action summary

trace_text = (
    site_action_register["Sparse_trace_alert_parameters"]
    .fillna("")
    .astype(str)
    .str.strip()
)

trace_alert_register = site_action_register.loc[trace_text.ne("")].copy()

low_evidence_register = site_action_register.loc[
    site_action_register["Chemical_parameters_tested"].le(2)
].copy()

p1_low_evidence_register = low_evidence_register.loc[
    low_evidence_register["Priority_tier"].astype(str).str.startswith("P1")
].copy()

tier_action_summary = (
    site_action_register
    .groupby("Priority_tier", dropna=False)
    .agg(
        Locations=("Testing_priority_score", "size"),
        Mean_priority_score=("Testing_priority_score", "mean"),
        Mean_recommended_tests=("Recommended_test_count", "mean"),
        Locations_with_targeted_retest=(
            "Targeted_retest_count", lambda x: x.gt(0).sum()
        ),
        Locations_with_baseline=(
            "Baseline_test_count", lambda x: x.gt(0).sum()
        ),
        Locations_with_trace_alert=(
            "Sparse_trace_alert_parameters",
            lambda x: x.fillna("").astype(str).str.strip().ne("").sum()
        )
    )
    .reset_index()
)

print("TRACE ALERT REGISTER")
print("Locations with separate trace alerts:", len(trace_alert_register))

print("\nLOW CHEMICAL-EVIDENCE REGISTER")
print("Locations with 0–2 measured core parameters:", len(low_evidence_register))
print("P1 locations in this group:", len(p1_low_evidence_register))

print("\nTIER ACTION SUMMARY")
display(tier_action_summary.round(2))

TRACE ALERT REGISTER
Locations with separate trace alerts: 150

LOW CHEMICAL-EVIDENCE REGISTER
Locations with 0–2 measured core parameters: 795
P1 locations in this group: 673

TIER ACTION SUMMARY


,Priority_tier,Locations,Mean_priority_score,Mean_recommended_tests,Locations_with_targeted_retest,Locations_with_baseline,Locations_with_trace_alert
0,P1 — Highest 10%,2421,37.01,8.00,195,2211,0
1,P2 — Next 15%,3629,32.86,8.00,493,3142,0
2,P3 — Remaining locations,18149,23.56,6.73,11363,6841,150


In [23]:
# CELL 19 — Export Phase 8 outputs

from pathlib import Path

PHASE8_DIR = (
    PROJECT_ROOT
    / "reports"
    / "testing_prioritization"
    / "phase_8"
)

PHASE8_DIR.mkdir(parents=True, exist_ok=True)


def export_register(df, filename):
    """Export a dataframe while retaining a spatial-group index when present."""
    out = df.copy()

    if "_spatial_group_id" not in out.columns:
        if out.index.name == "_spatial_group_id":
            out = out.reset_index()
        elif not isinstance(out.index, pd.RangeIndex):
            out = out.reset_index()

    path = PHASE8_DIR / filename
    out.to_csv(path, index=False)
    return path


saved_paths = {}

saved_paths["site_priority"] = export_register(
    priority_profile,
    "phase8_site_priority_register.csv"
)

saved_paths["site_actions"] = export_register(
    site_action_register,
    "phase8_site_action_register.csv"
)

saved_paths["parameter_actions"] = export_register(
    parameter_location_audit,
    "phase8_parameter_action_register.csv"
)

saved_paths["scenario_scores"] = export_register(
    priority_scenario_scores,
    "phase8_weight_sensitivity_scores.csv"
)

saved_paths["trace_alerts"] = export_register(
    trace_alert_register,
    "phase8_sparse_trace_alert_register.csv"
)

saved_paths["low_evidence"] = export_register(
    low_evidence_register,
    "phase8_low_chemical_evidence_register.csv"
)

saved_paths["p1_low_evidence"] = export_register(
    p1_low_evidence_register,
    "phase8_p1_low_evidence_register.csv"
)

saved_paths["tier_summary"] = export_register(
    tier_action_summary,
    "phase8_tier_action_summary.csv"
)


methodology_notes = """
BOREWELL SENTINEL INDIA — PHASE 8
INTELLIGENT TESTING PRIORITIZATION

Purpose:
Create an explainable pilot ranking of locations for groundwater testing.

Composite score weights:
- Chemical risk component: 40%
- Temporal trend component: 25%
- Monitoring gap component: 20%
- Data uncertainty component: 15%

These weights are expert-designed pilot assumptions, not learned or
independently validated weights.

Priority tiers:
- P1: highest 10% of ranked locations
- P2: next 15%
- P3: remaining locations

Interpretation:
The composite score represents testing and monitoring priority.
It is not a definitive contamination probability, water-safety verdict,
or substitute for laboratory testing.

Chemical evidence:
Locations with no valid measurements for a parameter are assigned
baseline testing recommendations. Lack of measurements must not be
interpreted as evidence of safe or contaminated water.

Trace alerts:
Sparse trace-element alerts are retained in separate review registers.
They are not included in the eight-parameter composite score.

Location identity:
Spatial group identifiers are provisional. Administrative labels and
coordinates require verification before field deployment.

Validation:
Phase 9 must evaluate sensitivity, threshold provenance, model and
ranking limitations, and the reliability of location-level conclusions.
"""

notes_path = PHASE8_DIR / "phase8_methodology_and_limitations.txt"
notes_path.write_text(methodology_notes.strip(), encoding="utf-8")
saved_paths["methodology"] = notes_path

print("PHASE 8 EXPORTS")
for name, path in saved_paths.items():
    print(f"{name}: {path}")

PHASE 8 EXPORTS
site_priority: c:\career\projects\BoreWell Sentinel India\reports\testing_prioritization\phase_8\phase8_site_priority_register.csv
site_actions: c:\career\projects\BoreWell Sentinel India\reports\testing_prioritization\phase_8\phase8_site_action_register.csv
parameter_actions: c:\career\projects\BoreWell Sentinel India\reports\testing_prioritization\phase_8\phase8_parameter_action_register.csv
scenario_scores: c:\career\projects\BoreWell Sentinel India\reports\testing_prioritization\phase_8\phase8_weight_sensitivity_scores.csv
trace_alerts: c:\career\projects\BoreWell Sentinel India\reports\testing_prioritization\phase_8\phase8_sparse_trace_alert_register.csv
low_evidence: c:\career\projects\BoreWell Sentinel India\reports\testing_prioritization\phase_8\phase8_low_chemical_evidence_register.csv
p1_low_evidence: c:\career\projects\BoreWell Sentinel India\reports\testing_prioritization\phase_8\phase8_p1_low_evidence_register.csv
tier_summary: c:\career\projects\BoreWell S

In [24]:
# CELL 20 — Final Phase 8 validation

expected_locations = 24_199
expected_parameter_rows = expected_locations * 8

assert len(priority_profile) == expected_locations
assert priority_profile.index.is_unique

assert len(site_action_register) == expected_locations
assert site_action_register.index.is_unique

assert len(parameter_location_audit) == expected_parameter_rows

assert site_action_register["Priority_rank"].notna().all()
assert site_action_register["Testing_priority_score"].notna().all()

assert (
    site_action_register["Recommended_test_count"].between(0, 8).all()
)

tier_counts = priority_profile["Priority_tier"].value_counts()

assert int(tier_counts.sum()) == expected_locations
assert int(tier_counts.get("P1 — Highest 10%", 0)) == 2_421
assert int(tier_counts.get("P2 — Next 15%", 0)) == 3_629
assert int(tier_counts.get("P3 — Remaining locations", 0)) == 18_149

missing_exports = [
    name for name, path in saved_paths.items()
    if not path.exists() or path.stat().st_size == 0
]

assert not missing_exports, f"Missing or empty exports: {missing_exports}"

print("PHASE 8 VALIDATION PASSED")
print("- Unique spatial locations:", len(priority_profile))
print("- Parameter-level action rows:", len(parameter_location_audit))
print("- Unique site action records:", site_action_register.index.nunique())
print("- P1 locations:", int(tier_counts.get("P1 — Highest 10%", 0)))
print("- P2 locations:", int(tier_counts.get("P2 — Next 15%", 0)))
print("- P3 locations:", int(tier_counts.get("P3 — Remaining locations", 0)))
print("- Trace-alert locations:", len(trace_alert_register))
print("- Low-evidence locations:", len(low_evidence_register))
print("- Verified output files:", len(saved_paths))
print("- Output directory:", PHASE8_DIR)

PHASE 8 VALIDATION PASSED
- Unique spatial locations: 24199
- Parameter-level action rows: 193592
- Unique site action records: 24199
- P1 locations: 2421
- P2 locations: 3629
- P3 locations: 18149
- Trace-alert locations: 150
- Low-evidence locations: 795
- Verified output files: 9
- Output directory: c:\career\projects\BoreWell Sentinel India\reports\testing_prioritization\phase_8


# Phase 8 — Intelligent Testing Prioritization: Completed

## Objective
Develop an explainable, location-level groundwater testing-priority
framework using chemical evidence, historical trends, monitoring gaps,
and data uncertainty.

## Final outputs
- Site-level testing priority register
- Location-level testing action register
- Parameter-level testing recommendations
- Weight sensitivity analysis
- Sparse trace-element alert register
- Low chemical-evidence register
- P1 low-evidence review register
- Priority-tier action summary
- Methodology and limitations documentation

## Final dataset coverage
- 174,635 source records retained in the integrated master dataset.
- 24,199 provisional spatial groups evaluated.
- 193,592 location–parameter action records generated.
- 2,421 P1, 3,629 P2, and 18,149 P3 locations ranked.

## Important limitations
- The scoring weights are expert-designed pilot assumptions.
- The composite score represents testing priority, not contamination probability.
- Sparse trace-element alerts are maintained separately.
- Provisional spatial identities and administrative labels require
  verification before operational field deployment.
- Historical sampling gaps and incomplete chemical evidence must not
  be interpreted as evidence of water safety.

## Status
**PHASE 8 COMPLETED**

Next: Phase 9 — Model Evaluation and Validation.